# Funding Data Integration and Quality

## Purpose

This notebook standardizes and integrates the processed NSF and CORDIS candidate datasets for use in GrantScopeAI.

NSF and CORDIS both describe funded research activities, so they can be combined into a shared grant-level table after their fields, data types, and definitions are aligned.

OpenAlex is also loaded for validation and topic-level comparison, but it remains a separate publication-context table because its analytical grain is one row per topic and year rather than one row per grant.

## Objectives

This notebook will:

- load the processed NSF, CORDIS, and OpenAlex datasets;
- compare the NSF and CORDIS schemas;
- define a common grant-level structure;
- transform both funding sources into the shared schema;
- combine the funding records into one integrated dataset;
- preserve source identifiers, currencies, URLs, and provenance;
- assign grants to a transparent shared topic taxonomy;
- create topic-year funding summaries;
- build a common machine-readable data-quality report;
- create a data dictionary for the final analytical tables;
- export GitHub-safe datasets for analysis, Tableau, and Streamlit.

## Important methodological boundaries

- NSF and CORDIS funding values remain in their native currencies.
- EUR and USD amounts are not added together or converted without an explicit exchange-rate methodology.
- OpenAlex records are not merged directly with individual grants.
- Topic assignments use transparent rule-based logic and may allow more than one topic per grant.
- Source-specific fields are preserved where they cannot be meaningfully standardized.

## Expected completion criteria

The notebook is complete when:

- 675 CORDIS projects and 2,664 NSF awards load successfully;
- the combined funding table contains 3,339 unique grant records;
- every grant has a globally unique `grant_key`;
- source counts reconcile with the input datasets;
- dates, currencies, relevance tiers, and URLs pass validation;
- the OpenAlex table remains separate with 40 topic-year records;
- topic-year funding summaries are created;
- a shared data-quality report and data dictionary are exported;
- the notebook runs from beginning to end without errors.

In [1]:
from pathlib import Path
import pandas as pd


# Relative paths from the Notebooks folder
PROCESSED_DATA_DIR = Path("../Data/Processed_Data")

CORDIS_PROCESSED_DIR = (
    PROCESSED_DATA_DIR
    / "CORDIS"
)

NSF_PROCESSED_DIR = (
    PROCESSED_DATA_DIR
    / "NSF"
)

OPENALEX_PROCESSED_DIR = (
    PROCESSED_DATA_DIR
    / "OpenAlex"
)


def find_latest_file(folder, pattern):
    """Return the most recently modified file matching a pattern."""

    matching_files = list(folder.glob(pattern))

    if not matching_files:
        raise FileNotFoundError(
            f"No file matching '{pattern}' was found in:\n{folder.resolve()}"
        )

    return max(
        matching_files,
        key=lambda path: path.stat().st_mtime
    )


# Locate the processed input files
cordis_path = find_latest_file(
    CORDIS_PROCESSED_DIR,
    "*candidate*.csv"
)

nsf_path = find_latest_file(
    NSF_PROCESSED_DIR,
    "nsf_candidate_awards_compact_*.csv"
)

openalex_path = find_latest_file(
    OPENALEX_PROCESSED_DIR,
    "openalex_topic_year_clean_*.csv"
)


# Load each source separately
cordis_df = pd.read_csv(
    cordis_path,
    low_memory=False
)

nsf_df = pd.read_csv(
    nsf_path,
    low_memory=False
)

openalex_df = pd.read_csv(
    openalex_path,
    low_memory=False
)


# Confirm which files were loaded
source_files = {
    "CORDIS": (cordis_path, cordis_df),
    "NSF": (nsf_path, nsf_df),
    "OpenAlex": (openalex_path, openalex_df)
}

for source_name, (file_path, dataframe) in source_files.items():

    print(f"\n{source_name}")
    print("-" * len(source_name))
    print("File:", file_path)
    print("Shape:", dataframe.shape)
    print("Columns:")
    print(dataframe.columns.tolist())


CORDIS
------
File: ..\Data\Processed_Data\CORDIS\cordis_candidate_projects_2021_2025.csv
Shape: (675, 75)
Columns:
['id', 'acronym', 'status', 'title', 'startDate', 'endDate', 'totalCost', 'ecMaxContribution', 'topics', 'ecSignatureDate', 'frameworkProgramme', 'masterCall', 'subCall', 'fundingScheme', 'nature', 'objective', 'contentUpdateDate', 'rcn', 'grantDoi', 'keywords', 'Human-validated', 'legalBasis', 'project_id_clean', 'record_year', 'in_project_date_scope', 'search_text', 'coordinator_organisation_id', 'coordinator_name', 'coordinator_short_name', 'coordinator_activity_type', 'coordinator_city', 'coordinator_country', 'coordinator_is_sme', 'coordinator_ec_contribution', 'coordinator_net_ec_contribution', 'coordinator_total_cost', 'organisation_count', 'country_count', 'countries', 'organisation_names', 'higher_education_count', 'private_company_count', 'research_organisation_count', 'public_body_count', 'sme_count', 'unknown_sme_count', 'coordinator_organisation_count', 'par

In [2]:
# Compare fields that may be used in the shared grant schema

cordis_integration_fields = [
    "project_id_clean",
    "title",
    "objective",
    "startDate",
    "endDate",
    "record_year",
    "totalCost",
    "ecMaxContribution",
    "coordinator_name",
    "coordinator_city",
    "coordinator_country",
    "call_topic_title",
    "fundingScheme",
    "status",
    "preferred_project_url",
    "relevance_tier"
]

nsf_integration_fields = [
    "source_id",
    "title",
    "abstract",
    "start_date",
    "end_date",
    "award_year",
    "amount_native",
    "amount_obligated",
    "currency",
    "organisation_name",
    "organisation_city",
    "organisation_country_code",
    "programme_name",
    "is_active_award",
    "source_url",
    "relevance_tier",
    "award_activity_type"
]


def build_field_summary(dataframe, fields, source_name):
    summary_rows = []

    for column in fields:
        summary_rows.append({
            "source": source_name,
            "column": column,
            "data_type": str(dataframe[column].dtype),
            "missing_count": dataframe[column].isna().sum(),
            "missing_pct": round(
                dataframe[column].isna().mean() * 100,
                2
            ),
            "unique_values": dataframe[column].nunique(
                dropna=True
            )
        })

    return pd.DataFrame(summary_rows)


cordis_field_summary_df = build_field_summary(
    cordis_df,
    cordis_integration_fields,
    "CORDIS"
)

nsf_field_summary_df = build_field_summary(
    nsf_df,
    nsf_integration_fields,
    "NSF"
)

integration_field_summary_df = pd.concat(
    [
        cordis_field_summary_df,
        nsf_field_summary_df
    ],
    ignore_index=True
)

display(integration_field_summary_df)

print("\nCORDIS funding and programme examples:")

display(
    cordis_df[
        [
            "project_id_clean",
            "totalCost",
            "ecMaxContribution",
            "call_topic_title",
            "fundingScheme",
            "status"
        ]
    ].head(10)
)

,source,column,data_type,missing_count,missing_pct,unique_values
0,CORDIS,project_id_clean,int64,0,0.0,675
1,CORDIS,title,object,0,0.0,674
2,CORDIS,objective,object,0,0.0,674
3,CORDIS,startDate,object,0,0.0,63
4,CORDIS,endDate,object,0,0.0,106
5,CORDIS,record_year,int64,0,0.0,4
6,CORDIS,totalCost,float64,0,0.0,374
7,CORDIS,ecMaxContribution,float64,0,0.0,543
8,CORDIS,coordinator_name,object,0,0.0,373
9,CORDIS,coordinator_city,object,0,0.0,285



CORDIS funding and programme examples:


,project_id_clean,totalCost,ecMaxContribution,call_topic_title,fundingScheme,status
0,101252959,61183298.75,29897910.0,Boosting innovation through better integration...,HORIZON-JU-RIA,SIGNED
1,101192848,19949864.80,19949437.4,Furthering the development of a materials acce...,HORIZON-RIA,SIGNED
2,101136570,15000000.00,15000000.0,Teaming for Excellence,HORIZON-CSA,SIGNED
3,101234224,30000000.00,15000000.0,Artificial Intelligence Factories,HORIZON-JU-RIA,SIGNED
4,101260391,30215955.00,14978231.0,Artificial Intelligence Factories,HORIZON-JU-RIA,SIGNED
5,101167540,13999999.00,13999999.0,ERC SYNERGY GRANTS,HORIZON-ERC-SYG,SIGNED
6,101167416,10968734.00,10968734.0,ERC SYNERGY GRANTS,HORIZON-ERC-SYG,SIGNED
7,101167045,9993698.00,9993698.0,ERC SYNERGY GRANTS,HORIZON-ERC-SYG,SIGNED
8,101167207,9991449.00,9991449.0,ERC SYNERGY GRANTS,HORIZON-ERC-SYG,SIGNED
9,101234349,19584750.00,9792375.0,Artificial Intelligence Factories,HORIZON-JU-RIA,SIGNED


In [3]:
# Compare the two CORDIS funding measures and programme fields

cordis_equal_amounts = (
    cordis_df["ecMaxContribution"]
    .eq(cordis_df["totalCost"])
    .sum()
)

cordis_eu_below_total = (
    cordis_df["ecMaxContribution"]
    .lt(cordis_df["totalCost"])
    .sum()
)

cordis_eu_above_total = (
    cordis_df["ecMaxContribution"]
    .gt(cordis_df["totalCost"])
    .sum()
)

cordis_funding_summary_df = pd.DataFrame({
    "funding_field": [
        "totalCost",
        "ecMaxContribution"
    ],
    "minimum": [
        cordis_df["totalCost"].min(),
        cordis_df["ecMaxContribution"].min()
    ],
    "median": [
        cordis_df["totalCost"].median(),
        cordis_df["ecMaxContribution"].median()
    ],
    "maximum": [
        cordis_df["totalCost"].max(),
        cordis_df["ecMaxContribution"].max()
    ],
    "missing_count": [
        cordis_df["totalCost"].isna().sum(),
        cordis_df["ecMaxContribution"].isna().sum()
    ]
})

cordis_funding_scheme_counts_df = (
    cordis_df["fundingScheme"]
    .value_counts()
    .rename_axis("fundingScheme")
    .reset_index(name="project_count")
)

print("Funding-field comparison:")
print(
    "Equal EU contribution and total cost:",
    f"{cordis_equal_amounts:,}"
)
print(
    "EU contribution below total cost:",
    f"{cordis_eu_below_total:,}"
)
print(
    "EU contribution above total cost:",
    f"{cordis_eu_above_total:,}"
)

display(cordis_funding_summary_df)

print("\nMost common funding schemes:")
display(cordis_funding_scheme_counts_df.head(15))

print("\nExample programme and funding-scheme combinations:")
display(
    cordis_df[
        [
            "project_id_clean",
            "call_topic_title",
            "fundingScheme",
            "totalCost",
            "ecMaxContribution"
        ]
    ]
    .drop_duplicates(
        subset=[
            "call_topic_title",
            "fundingScheme"
        ]
    )
    .head(20)
)

Funding-field comparison:
Equal EU contribution and total cost: 273
EU contribution below total cost: 132
EU contribution above total cost: 270


,funding_field,minimum,median,maximum,missing_count
0,totalCost,0.0,1944825.0,61183298.75,0
1,ecMaxContribution,75000.0,2495850.0,37995202.50,0



Most common funding schemes:


,fundingScheme,project_count
0,HORIZON-ERC,151
1,HORIZON-RIA,146
2,HORIZON-TMA-MSCA-PF-EF,111
3,HORIZON-EIC,49
4,HORIZON-IA,38
5,HORIZON-ERC-POC,31
6,HORIZON-TMA-MSCA-DN,28
7,HORIZON-CSA,24
8,HORIZON-JU-RIA,20
9,HORIZON-EIC-ACC-BF,18



Example programme and funding-scheme combinations:


,project_id_clean,call_topic_title,fundingScheme,totalCost,ecMaxContribution
0,101252959,Boosting innovation through better integration...,HORIZON-JU-RIA,61183298.75,29897910.00
1,101192848,Furthering the development of a materials acce...,HORIZON-RIA,19949864.80,19949437.40
2,101136570,Teaming for Excellence,HORIZON-CSA,15000000.00,15000000.00
3,101234224,Artificial Intelligence Factories,HORIZON-JU-RIA,30000000.00,15000000.00
5,101167540,ERC SYNERGY GRANTS,HORIZON-ERC-SYG,13999999.00,13999999.00
10,101177877,Advanced biomaterials for the Health Care (IA),HORIZON-IA,0.00,8649981.64
11,101178389,Biodegradable polymers for sustainable packagi...,HORIZON-IA,8551956.25,7947739.76
12,101184736,"Inland ice, including snow cover, glaciers, ic...",HORIZON-RIA,7820258.75,7820258.75
13,101138031,Battery management system (BMS) and battery sy...,HORIZON-IA,0.00,7477609.93
14,101187940,Innovative and customizable services for EOSC ...,HORIZON-RIA,6999997.50,6999997.50


## Defining the shared funding schema

NSF and CORDIS both describe funded research activities, but their original field names and definitions differ. A documented schema crosswalk is therefore created before combining the records.

The main integration decisions are:

- one row represents one unique funded award or project;
- source-prefixed `grant_key` values provide globally unique identifiers;
- the CORDIS `objective` field is aligned with the NSF award abstract;
- award year is defined consistently as the year of the standardized start date;
- CORDIS `ecMaxContribution` is used as its primary funding amount;
- CORDIS `totalCost` is preserved separately as project total cost;
- NSF `estimatedTotalAmt` remains its primary funding amount;
- `call_topic_title` is used as the CORDIS equivalent of the NSF programme field;
- `fundingScheme` remains a separate CORDIS funding-mechanism field;
- native currencies are preserved as EUR and USD.

The two currencies are not converted or added together because no exchange-rate methodology has been defined. Source-specific fields are retained where their meanings are not genuinely comparable.

In [4]:
# Define the shared NSF–CORDIS grant schema

schema_crosswalk_rows = [
    {
        "standard_column": "source",
        "cordis_field": "constant: CORDIS",
        "nsf_field": "source",
        "definition": "Original funding data source"
    },
    {
        "standard_column": "grant_key",
        "cordis_field": "CORDIS_ + project_id_clean",
        "nsf_field": "grant_key",
        "definition": "Globally unique source-prefixed grant identifier"
    },
    {
        "standard_column": "source_id",
        "cordis_field": "project_id_clean",
        "nsf_field": "source_id",
        "definition": "Original source-specific award or project identifier"
    },
    {
        "standard_column": "title",
        "cordis_field": "title",
        "nsf_field": "title",
        "definition": "Grant or project title"
    },
    {
        "standard_column": "abstract",
        "cordis_field": "objective",
        "nsf_field": "abstract",
        "definition": "Main descriptive text"
    },
    {
        "standard_column": "start_date",
        "cordis_field": "startDate",
        "nsf_field": "start_date",
        "definition": "Grant or project start date"
    },
    {
        "standard_column": "end_date",
        "cordis_field": "endDate",
        "nsf_field": "end_date",
        "definition": "Grant or project end date"
    },
    {
        "standard_column": "award_year",
        "cordis_field": "record_year",
        "nsf_field": "award_year",
        "definition": "Year the funded activity began"
    },
    {
        "standard_column": "amount_native",
        "cordis_field": "ecMaxContribution",
        "nsf_field": "amount_native",
        "definition": "Primary award value in the source currency"
    },
    {
        "standard_column": "currency",
        "cordis_field": "constant: EUR",
        "nsf_field": "currency",
        "definition": "Native currency of the primary amount"
    },
    {
        "standard_column": "organisation_name",
        "cordis_field": "coordinator_name",
        "nsf_field": "organisation_name",
        "definition": "Lead or recipient organisation"
    },
    {
        "standard_column": "organisation_city",
        "cordis_field": "coordinator_city",
        "nsf_field": "organisation_city",
        "definition": "City of the lead or recipient organisation"
    },
    {
        "standard_column": "organisation_country",
        "cordis_field": "coordinator_country",
        "nsf_field": "organisation_country_code",
        "definition": "Country or country code reported by the source"
    },
    {
        "standard_column": "programme_name",
        "cordis_field": "call_topic_title",
        "nsf_field": "programme_name",
        "definition": "Funding programme or call topic"
    },
    {
        "standard_column": "funding_mechanism",
        "cordis_field": "fundingScheme",
        "nsf_field": "not available",
        "definition": "Source-specific funding scheme or mechanism"
    },
    {
        "standard_column": "status",
        "cordis_field": "status",
        "nsf_field": "derived from is_active_award",
        "definition": "Source-reported or derived award status"
    },
    {
        "standard_column": "source_url",
        "cordis_field": "preferred_project_url",
        "nsf_field": "source_url",
        "definition": "Link to the original funding record"
    },
    {
        "standard_column": "relevance_tier",
        "cordis_field": "relevance_tier",
        "nsf_field": "relevance_tier",
        "definition": "Core or broad relevance classification"
    },
    {
        "standard_column": "activity_type",
        "cordis_field": "nature",
        "nsf_field": "award_activity_type",
        "definition": "Source-specific funded activity classification"
    }
]

schema_crosswalk_df = pd.DataFrame(schema_crosswalk_rows)

display(schema_crosswalk_df)

print(
    "Shared schema columns:",
    len(schema_crosswalk_df)
)

,standard_column,cordis_field,nsf_field,definition
0,source,constant: CORDIS,source,Original funding data source
1,grant_key,CORDIS_ + project_id_clean,grant_key,Globally unique source-prefixed grant identifier
2,source_id,project_id_clean,source_id,Original source-specific award or project iden...
3,title,title,title,Grant or project title
4,abstract,objective,abstract,Main descriptive text
5,start_date,startDate,start_date,Grant or project start date
6,end_date,endDate,end_date,Grant or project end date
7,award_year,record_year,award_year,Year the funded activity began
8,amount_native,ecMaxContribution,amount_native,Primary award value in the source currency
9,currency,constant: EUR,currency,Native currency of the primary amount


Shared schema columns: 19


In [5]:
# Compare status and activity classifications before standardization

cordis_nature_counts_df = (
    cordis_df["nature"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("cordis_nature")
    .reset_index(name="project_count")
)

nsf_activity_counts_df = (
    nsf_df["award_activity_type"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("nsf_activity_type")
    .reset_index(name="award_count")
)

cordis_status_counts_df = (
    cordis_df["status"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("cordis_status")
    .reset_index(name="project_count")
)

nsf_status_counts_df = (
    nsf_df["is_active_award"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("is_active_award")
    .reset_index(name="award_count")
)

print("CORDIS activity values:")
display(cordis_nature_counts_df)

print("\nNSF activity values:")
display(nsf_activity_counts_df)

print("\nCORDIS status values:")
display(cordis_status_counts_df)

print("\nNSF active-status values:")
display(nsf_status_counts_df)

CORDIS activity values:


,cordis_nature,project_count
0,Missing,675



NSF activity values:


,nsf_activity_type,award_count
0,research,2370
1,commercialisation,136
2,conference_or_workshop,113
3,infrastructure,45



CORDIS status values:


,cordis_status,project_count
0,SIGNED,621
1,CLOSED,34
2,TERMINATED,20



NSF active-status values:


,is_active_award,award_count
0,True,1805
1,False,859


In [7]:
# Correct the activity-type mapping after inspecting CORDIS nature

schema_crosswalk_df.loc[
    schema_crosswalk_df["standard_column"].eq("activity_type"),
    [
        "cordis_field",
        "nsf_field",
        "definition"
    ]
] = [
    "not available",
    "award_activity_type",
    (
        "Standardized activity category where available; "
        "currently provided only by NSF"
    )
]

# Add a separate source-specific activity field
source_activity_row = pd.DataFrame([
    {
        "standard_column": "source_activity_type",
        "cordis_field": "nature",
        "nsf_field": "award_activity_type",
        "definition": (
            "Original source-specific activity classification; "
            "CORDIS nature is missing for all candidate projects"
        )
    }
])

schema_crosswalk_df = pd.concat(
    [
        schema_crosswalk_df,
        source_activity_row
    ],
    ignore_index=True
)

display(
    schema_crosswalk_df.loc[
        schema_crosswalk_df["standard_column"].isin(
            [
                "funding_mechanism",
                "activity_type",
                "source_activity_type"
            ]
        )
    ]
)

,standard_column,cordis_field,nsf_field,definition
14,funding_mechanism,fundingScheme,not available,Source-specific funding scheme or mechanism
18,activity_type,not available,award_activity_type,Standardized activity category where available...
19,source_activity_type,nature,award_activity_type,Original source-specific activity classificati...


## Standardizing CORDIS project records

The CORDIS candidate dataset contains 675 unique projects, but its original fields must be transformed before it can be appended to NSF.

The following decisions are applied:

- `project_id_clean` becomes the source-specific identifier;
- `CORDIS_` is added to create a globally unique `grant_key`;
- `objective` becomes the standardized abstract field;
- ISO-formatted project dates are parsed using the explicit `%Y-%m-%d` format;
- award year is derived from the parsed start date and checked against `record_year`;
- `ecMaxContribution` becomes the primary funding amount;
- `totalCost` is retained separately because it represents total project cost rather than awarded EU funding;
- coordinator information represents the lead organisation;
- `call_topic_title` becomes the standardized programme field;
- `fundingScheme` is retained as the source-specific funding mechanism;
- CORDIS status values are preserved in their original form;
- currency is assigned as EUR.

The CORDIS `nature` field is missing for all 675 candidate projects. No activity category is inferred from the funding scheme, so the standardized activity fields remain missing rather than introducing unsupported classifications.

In [9]:
# Transform CORDIS into the shared grant schema

def clean_string(series):
    return (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )


cordis_source_id = (
    cordis_df["project_id_clean"]
    .astype("Int64")
    .astype("string")
)

cordis_start_date = pd.to_datetime(
    cordis_df["startDate"],
    format="%Y-%m-%d",
    errors="coerce"
)

cordis_end_date = pd.to_datetime(
    cordis_df["endDate"],
    format="%Y-%m-%d",
    errors="coerce"
)


cordis_standard_df = pd.DataFrame({
    "source": "CORDIS",

    "grant_key": (
        "CORDIS_" + cordis_source_id
    ),

    "source_id": cordis_source_id,

    "title": clean_string(
        cordis_df["title"]
    ),

    "abstract": clean_string(
        cordis_df["objective"]
    ),

    "start_date": cordis_start_date,

    "end_date": cordis_end_date,

    # Derive the analytical year directly from the start date
    "award_year": (
        cordis_start_date.dt.year.astype("Int64")
    ),

    # Maximum EU contribution is the primary funding amount
    "amount_native": pd.to_numeric(
        cordis_df["ecMaxContribution"],
        errors="coerce"
    ),

    # Preserve total project cost separately
    "project_total_cost": pd.to_numeric(
        cordis_df["totalCost"],
        errors="coerce"
    ),

    # NSF-specific field, unavailable for CORDIS
    "amount_obligated": pd.Series(
        pd.NA,
        index=cordis_df.index,
        dtype="Float64"
    ),

    "currency": "EUR",

    "organisation_name": clean_string(
        cordis_df["coordinator_name"]
    ),

    "organisation_city": clean_string(
        cordis_df["coordinator_city"]
    ),

    "organisation_country": (
        clean_string(
            cordis_df["coordinator_country"]
        )
        .str.upper()
    ),

    "programme_name": clean_string(
        cordis_df["call_topic_title"]
    ),

    "funding_mechanism": clean_string(
        cordis_df["fundingScheme"]
    ),

    "status": (
        clean_string(
            cordis_df["status"]
        )
        .str.upper()
    ),

    # CORDIS nature is missing for all candidate projects
    "activity_type": pd.Series(
        pd.NA,
        index=cordis_df.index,
        dtype="string"
    ),

    "source_activity_type": pd.Series(
        pd.NA,
        index=cordis_df.index,
        dtype="string"
    ),

    "source_url": clean_string(
        cordis_df["preferred_project_url"]
    ),

    "relevance_tier": clean_string(
        cordis_df["relevance_tier"]
    )
})


cordis_year_mismatch_count = (
    cordis_standard_df["award_year"]
    .ne(
        cordis_df["record_year"].astype("Int64")
    )
    .fillna(False)
    .sum()
)


print(
    "Standardized CORDIS rows:",
    f"{len(cordis_standard_df):,}"
)

print(
    "Duplicate grant keys:",
    cordis_standard_df["grant_key"]
    .duplicated()
    .sum()
)

print(
    "Invalid start dates:",
    cordis_standard_df["start_date"]
    .isna()
    .sum()
)

print(
    "Invalid end dates:",
    cordis_standard_df["end_date"]
    .isna()
    .sum()
)

print(
    "Start-year disagreements with record_year:",
    cordis_year_mismatch_count
)

print(
    "Missing primary amounts:",
    cordis_standard_df["amount_native"]
    .isna()
    .sum()
)

print(
    "Currencies:",
    cordis_standard_df["currency"]
    .unique()
    .tolist()
)

display(cordis_standard_df.head())

Standardized CORDIS rows: 675
Duplicate grant keys: 0
Invalid start dates: 0
Invalid end dates: 0
Start-year disagreements with record_year: 0
Missing primary amounts: 0
Currencies: ['EUR']


,source,grant_key,source_id,title,abstract,start_date,end_date,award_year,amount_native,project_total_cost,...,organisation_name,organisation_city,organisation_country,programme_name,funding_mechanism,status,activity_type,source_activity_type,source_url,relevance_tier
0,CORDIS,CORDIS_101252959,101252959,Protein-ligand data generation at scale to sup...,The ability to discover and optimize small-mol...,2025-11-01,2030-10-31,2025,29897910.0,61183298.75,...,STRUCTURAL GENOMICS CONSORTIUM LBG,London,UK,Boosting innovation through better integration...,HORIZON-JU-RIA,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101252959,core_match
1,CORDIS,CORDIS_101192848,101192848,"FULLy integrated, autonomous & chemistry agnos...",Battery technology emerges as a key solution f...,2025-02-01,2029-01-31,2025,19949437.4,19949864.80,...,VRIJE UNIVERSITEIT BRUSSEL,BRUSSEL,BE,Furthering the development of a materials acce...,HORIZON-RIA,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101192848,core_match
2,CORDIS,CORDIS_101136570,101136570,Translational Research and Innovation in Ophth...,Eye diseases affect the daily lives of 2.2 bil...,2025-01-01,2030-12-31,2025,15000000.0,15000000.00,...,INSTYTUT CHEMII FIZYCZNEJ POLSKIEJ AKADEMII NAUK,Warszawa,PL,Teaming for Excellence,HORIZON-CSA,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101136570,core_match
3,CORDIS,CORDIS_101234224,101234224,Italy for Artificial Intelligence,IT4LIA AI Factory will act as a catalyst for t...,2025-04-01,2028-03-31,2025,15000000.0,30000000.00,...,CINECA CONSORZIO INTERUNIVERSITARIO,CASALECCHIO DI RENO BO,IT,Artificial Intelligence Factories,HORIZON-JU-RIA,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101234224,core_match
4,CORDIS,CORDIS_101260391,101260391,AI Factory France,"""Building upon France’s National AI Strategy l...",2025-10-01,2028-09-30,2025,14978231.0,30215955.00,...,GRAND EQUIPEMENT NATIONAL DE CALCUL INTENSIF,Paris,FR,Artificial Intelligence Factories,HORIZON-JU-RIA,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101260391,core_match


## Standardizing NSF award records

The NSF candidate dataset already uses a compact application-ready structure, but several fields still need to be aligned with the CORDIS schema.

The following transformations are applied:

- the existing source-prefixed `grant_key` is retained;
- `source_id` remains the original NSF award identifier;
- NSF start and end dates are parsed using the explicit `%Y-%m-%d` format;
- award year is derived from the parsed start date and checked against the original `award_year`;
- `estimatedTotalAmt`, previously standardized as `amount_native`, remains the primary award value;
- obligated funding is preserved separately as `amount_obligated`;
- currency remains USD;
- the recipient institution becomes the standardized organisation;
- the cleaned NSF programme field becomes `programme_name`;
- the Boolean active-award field is converted to `ACTIVE` or `INACTIVE`;
- NSF award activity labels are retained in both the standardized and source-specific activity fields.

NSF does not provide a directly comparable field for CORDIS project total cost or funding mechanism. These fields therefore remain missing rather than being inferred from unrelated programme information.

In [10]:
# Transform NSF into the shared grant schema

nsf_start_date = pd.to_datetime(
    nsf_df["start_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

nsf_end_date = pd.to_datetime(
    nsf_df["end_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

nsf_source_id = (
    nsf_df["source_id"]
    .astype("Int64")
    .astype("string")
)

nsf_status = (
    nsf_df["is_active_award"]
    .map({
        True: "ACTIVE",
        False: "INACTIVE"
    })
    .astype("string")
)


nsf_standard_df = pd.DataFrame({
    "source": "NSF",

    "grant_key": clean_string(
        nsf_df["grant_key"]
    ),

    "source_id": nsf_source_id,

    "title": clean_string(
        nsf_df["title"]
    ),

    "abstract": clean_string(
        nsf_df["abstract"]
    ),

    "start_date": nsf_start_date,

    "end_date": nsf_end_date,

    "award_year": (
        nsf_start_date.dt.year.astype("Int64")
    ),

    "amount_native": pd.to_numeric(
        nsf_df["amount_native"],
        errors="coerce"
    ),

    # CORDIS-specific field, unavailable for NSF
    "project_total_cost": pd.Series(
        pd.NA,
        index=nsf_df.index,
        dtype="Float64"
    ),

    "amount_obligated": pd.to_numeric(
        nsf_df["amount_obligated"],
        errors="coerce"
    ),

    "currency": (
        clean_string(
            nsf_df["currency"]
        )
        .str.upper()
    ),

    "organisation_name": clean_string(
        nsf_df["organisation_name"]
    ),

    "organisation_city": clean_string(
        nsf_df["organisation_city"]
    ),

    "organisation_country": (
        clean_string(
            nsf_df["organisation_country_code"]
        )
        .str.upper()
    ),

    "programme_name": clean_string(
        nsf_df["programme_name"]
    ),

    # No directly comparable NSF funding-mechanism field
    "funding_mechanism": pd.Series(
        pd.NA,
        index=nsf_df.index,
        dtype="string"
    ),

    "status": nsf_status,

    "activity_type": clean_string(
        nsf_df["award_activity_type"]
    ),

    "source_activity_type": clean_string(
        nsf_df["award_activity_type"]
    ),

    "source_url": clean_string(
        nsf_df["source_url"]
    ),

    "relevance_tier": clean_string(
        nsf_df["relevance_tier"]
    )
})


nsf_year_mismatch_count = (
    nsf_standard_df["award_year"]
    .ne(
        nsf_df["award_year"].astype("Int64")
    )
    .fillna(False)
    .sum()
)


print(
    "Standardized NSF rows:",
    f"{len(nsf_standard_df):,}"
)

print(
    "Duplicate grant keys:",
    nsf_standard_df["grant_key"]
    .duplicated()
    .sum()
)

print(
    "Invalid start dates:",
    nsf_standard_df["start_date"]
    .isna()
    .sum()
)

print(
    "Invalid end dates:",
    nsf_standard_df["end_date"]
    .isna()
    .sum()
)

print(
    "Start-year disagreements with award_year:",
    nsf_year_mismatch_count
)

print(
    "Missing primary amounts:",
    nsf_standard_df["amount_native"]
    .isna()
    .sum()
)

print(
    "Currencies:",
    nsf_standard_df["currency"]
    .dropna()
    .unique()
    .tolist()
)

print(
    "Column order matches CORDIS:",
    nsf_standard_df.columns.tolist()
    == cordis_standard_df.columns.tolist()
)

display(nsf_standard_df.head())

Standardized NSF rows: 2,664
Duplicate grant keys: 0
Invalid start dates: 0
Invalid end dates: 0
Start-year disagreements with award_year: 0
Missing primary amounts: 0
Currencies: ['USD']
Column order matches CORDIS: True


,source,grant_key,source_id,title,abstract,start_date,end_date,award_year,amount_native,project_total_cost,...,organisation_name,organisation_city,organisation_country,programme_name,funding_mechanism,status,activity_type,source_activity_type,source_url,relevance_tier
0,NSF,NSF_2433348,2433348,AI-Materials Institute (AI-MI),The need for materials with improved or new pr...,2025-10-01,2030-09-30,2025,20000000.0,<NA>,...,Cornell University,ITHACA,US,"NSF-Intel Semiconductr Partnrs, AI Research In...",<NA>,ACTIVE,research,research,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match
1,NSF,NSF_2445868,2445868,"MIP: Biomaterials, Polymers, and Advanced Cons...","The BioPACIFIC MIP (Biomaterials, Polymers and...",2025-08-01,2030-07-31,2025,19800000.0,<NA>,...,University of California-Santa Barbara,SANTA BARBARA,US,Materials Innovation Platforms,<NA>,ACTIVE,research,research,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match
2,NSF,NSF_2503773,2503773,Center: NSF Center for Synthetic Organic Elect...,The ability to use electrons as chemical reage...,2025-09-01,2030-08-31,2025,19800000.0,<NA>,...,Missouri University of Science and Technology,ROLLA,US,CHE CENTERS,<NA>,ACTIVE,research,research,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match
3,NSF,NSF_2503933,2503933,NSF Center for Single-Entity Nanochemistry and...,The NSF Center for Single-Entity Nanochemistry...,2025-09-01,2030-08-31,2025,19800000.0,<NA>,...,Indiana University,BLOOMINGTON,US,CHE CENTERS,<NA>,ACTIVE,research,research,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match
4,NSF,NSF_2503885,2503885,NSF Center for Genetically Encoded Materials,The Center for Genetically Encoded Materials (...,2025-09-01,2030-08-31,2025,18000000.0,<NA>,...,University of California-Berkeley,BERKELEY,US,CHE CENTERS,<NA>,ACTIVE,research,research,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match


## Aligning and combining the funding sources

Before appending NSF and CORDIS, the standardized columns are assigned consistent data types.

This prevents source-specific missing columns from changing the inferred data types during concatenation. In particular:

- shared text fields use pandas string types;
- start and end dates use datetime types;
- award year uses a nullable integer type;
- funding fields use nullable floating-point types;
- source-specific fields remain missing where they are unavailable.

NSF and CORDIS are then combined using a row-wise append rather than a join. The two sources describe the same general analytical unit—a funded award or project—but they do not share identifiers and should not be matched directly to one another.

The combined table is validated by confirming:

- 675 CORDIS projects and 2,664 NSF awards are retained;
- the final table contains 3,339 records;
- source counts reconcile with the input datasets;
- every `grant_key` remains globally unique;
- no records contain missing or unexpected source labels.

In [12]:
# Align shared data types before combining the funding sources

shared_string_columns = [
    "source",
    "grant_key",
    "source_id",
    "title",
    "abstract",
    "currency",
    "organisation_name",
    "organisation_city",
    "organisation_country",
    "programme_name",
    "funding_mechanism",
    "status",
    "activity_type",
    "source_activity_type",
    "source_url",
    "relevance_tier"
]

shared_float_columns = [
    "amount_native",
    "project_total_cost",
    "amount_obligated"
]

shared_date_columns = [
    "start_date",
    "end_date"
]


for dataframe in [
    cordis_standard_df,
    nsf_standard_df
]:

    for column in shared_string_columns:
        dataframe[column] = (
            dataframe[column]
            .astype("string")
        )

    for column in shared_float_columns:
        dataframe[column] = (
            pd.to_numeric(
                dataframe[column],
                errors="coerce"
            )
            .astype("Float64")
        )

    for column in shared_date_columns:
        dataframe[column] = pd.to_datetime(
            dataframe[column],
            errors="coerce"
        )

    dataframe["award_year"] = (
        pd.to_numeric(
            dataframe["award_year"],
            errors="coerce"
        )
        .astype("Int64")
    )


dtype_alignment_df = pd.DataFrame({
    "column": cordis_standard_df.columns,
    "cordis_dtype": [
        str(dtype)
        for dtype in cordis_standard_df.dtypes
    ],
    "nsf_dtype": [
        str(dtype)
        for dtype in nsf_standard_df.dtypes
    ]
})

dtype_alignment_df["types_match"] = (
    dtype_alignment_df["cordis_dtype"]
    .eq(dtype_alignment_df["nsf_dtype"])
)

print(
    "Columns with matching data types:",
    dtype_alignment_df["types_match"].sum(),
    "of",
    len(dtype_alignment_df)
)

display(
    dtype_alignment_df.loc[
        ~dtype_alignment_df["types_match"]
    ]
)

Columns with matching data types: 22 of 22


,column,cordis_dtype,nsf_dtype,types_match


In [13]:
# Combine the standardized funding sources

grants_clean_df = pd.concat(
    [
        cordis_standard_df,
        nsf_standard_df
    ],
    ignore_index=True
)

grants_clean_df = (
    grants_clean_df
    .sort_values(
        [
            "source",
            "award_year",
            "grant_key"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


source_count_summary_df = (
    grants_clean_df["source"]
    .value_counts()
    .rename_axis("source")
    .reset_index(name="grant_count")
    .sort_values("source")
)

expected_combined_rows = (
    len(cordis_standard_df)
    + len(nsf_standard_df)
)

actual_combined_rows = len(grants_clean_df)

duplicate_global_keys = (
    grants_clean_df["grant_key"]
    .duplicated()
    .sum()
)

missing_source_values = (
    grants_clean_df["source"]
    .isna()
    .sum()
)

unexpected_sources = (
    ~grants_clean_df["source"]
    .isin(["CORDIS", "NSF"])
).sum()


print(
    "Expected combined rows:",
    f"{expected_combined_rows:,}"
)

print(
    "Actual combined rows:",
    f"{actual_combined_rows:,}"
)

print(
    "Row counts reconcile:",
    actual_combined_rows == expected_combined_rows
)

print(
    "Duplicate global grant keys:",
    duplicate_global_keys
)

print(
    "Missing source values:",
    missing_source_values
)

print(
    "Unexpected source values:",
    unexpected_sources
)

print("\nRows by source:")
display(source_count_summary_df)

display(grants_clean_df.head())

Expected combined rows: 3,339
Actual combined rows: 3,339
Row counts reconcile: True
Duplicate global grant keys: 0
Missing source values: 0
Unexpected source values: 0

Rows by source:


,source,grant_count
1,CORDIS,675
0,NSF,2664


,source,grant_key,source_id,title,abstract,start_date,end_date,award_year,amount_native,project_total_cost,...,organisation_name,organisation_city,organisation_country,programme_name,funding_mechanism,status,activity_type,source_activity_type,source_url,relevance_tier
0,CORDIS,CORDIS_101039636,101039636,"Piezoelectric Biomolecules for lead-free, Reli...",Billions of piezoelectric sensors are produced...,2022-06-01,2027-05-31,2022,1499525.0,1499525.0,...,UNIVERSITY OF LIMERICK,Limerick,IE,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101039636,core_match
1,CORDIS,CORDIS_101040353,101040353,Exploring the Molecular Properties of Atmosphe...,Aerosol particles are ubiquitous constituents ...,2022-04-01,2027-03-31,2022,1462491.0,1462491.0,...,AARHUS UNIVERSITET,Aarhus C,DK,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101040353,core_match
2,CORDIS,CORDIS_101040355,101040355,Probing (Orphan) Nuclear Receptors in Neurodeg...,"Neurodegenerative diseases such as Alzheimers,...",2022-05-01,2027-04-30,2022,1498813.0,1498813.0,...,LUDWIG-MAXIMILIANS-UNIVERSITAET MUENCHEN,Planegg,DE,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101040355,core_match
3,CORDIS,CORDIS_101040729,101040729,FIrst NEar-TErm ApplicationS of QUAntum Devices,Quantum technologies have set remarkable miles...,2022-05-01,2027-04-30,2022,1485042.5,1485042.5,...,UNIVERSITEIT LEIDEN,Leiden,NL,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101040729,core_match
4,CORDIS,CORDIS_101041177,101041177,Deciphering cellular and viral determinants of...,Herpes simplex virus 1 (HSV-1) is an important...,2022-05-01,2028-04-30,2022,1998008.75,1998008.75,...,MEDIZINISCHE HOCHSCHULE HANNOVER,Hannover,DE,ERC CONSOLIDATOR GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101041177,broad_match


In [14]:
# Validate shared-field completeness and integration rules

core_integration_fields = [
    "grant_key",
    "source_id",
    "title",
    "abstract",
    "start_date",
    "end_date",
    "award_year",
    "amount_native",
    "currency",
    "organisation_name",
    "organisation_city",
    "organisation_country",
    "programme_name",
    "status",
    "source_url",
    "relevance_tier"
]


integration_quality_rows = []

for source_name, source_df in grants_clean_df.groupby(
    "source",
    dropna=False
):

    for column in core_integration_fields:

        missing_count = source_df[column].isna().sum()

        integration_quality_rows.append({
            "source": source_name,
            "column": column,
            "records_checked": len(source_df),
            "missing_count": missing_count,
            "missing_pct": round(
                missing_count / len(source_df) * 100,
                2
            ),
            "unique_values": source_df[column].nunique(
                dropna=True
            )
        })


integration_field_quality_df = pd.DataFrame(
    integration_quality_rows
)


outside_date_scope_count = (
    ~grants_clean_df["award_year"].between(
        2021,
        2025
    )
).sum()

negative_amount_count = (
    grants_clean_df["amount_native"]
    .lt(0)
    .sum()
)

invalid_url_count = (
    ~grants_clean_df["source_url"]
    .fillna("")
    .str.startswith(("http://", "https://"))
).sum()

unexpected_relevance_count = (
    ~grants_clean_df["relevance_tier"]
    .isin([
        "core_match",
        "broad_match"
    ])
).sum()

invalid_currency_count = (
    (
        grants_clean_df["source"].eq("CORDIS")
        & grants_clean_df["currency"].ne("EUR")
    )
    |
    (
        grants_clean_df["source"].eq("NSF")
        & grants_clean_df["currency"].ne("USD")
    )
).sum()


print(
    "Awards outside 2021–2025:",
    outside_date_scope_count
)

print(
    "Negative primary amounts:",
    negative_amount_count
)

print(
    "Invalid source URLs:",
    invalid_url_count
)

print(
    "Unexpected relevance tiers:",
    unexpected_relevance_count
)

print(
    "Source–currency mismatches:",
    invalid_currency_count
)

print("\nShared-field completeness by source:")

display(
    integration_field_quality_df.sort_values(
        [
            "missing_pct",
            "source",
            "column"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)

Awards outside 2021–2025: 0
Negative primary amounts: 0
Invalid source URLs: 0
Unexpected relevance tiers: 0
Source–currency mismatches: 0

Shared-field completeness by source:


,source,column,records_checked,missing_count,missing_pct,unique_values
3,CORDIS,abstract,675,0,0.0,674
7,CORDIS,amount_native,675,0,0.0,543
6,CORDIS,award_year,675,0,0.0,4
8,CORDIS,currency,675,0,0.0,1
5,CORDIS,end_date,675,0,0.0,106
0,CORDIS,grant_key,675,0,0.0,675
10,CORDIS,organisation_city,675,0,0.0,285
11,CORDIS,organisation_country,675,0,0.0,34
9,CORDIS,organisation_name,675,0,0.0,373
12,CORDIS,programme_name,675,0,0.0,223


## Defining the shared topic taxonomy

A common topic vocabulary is required to compare NSF and CORDIS funding activity with OpenAlex publication trends.

The taxonomy uses the same eight topics already represented in the OpenAlex dataset:

- Autonomous laboratories;
- Reaction prediction;
- AI-enabled catalysis;
- Materials informatics;
- Cheminformatics;
- Molecular machine learning;
- AI-enabled materials;
- AI-enabled chemistry.

Each topic is defined using transparent keyword rules applied to grant titles, abstracts, programme names, and funding-mechanism text.

The rules are ordered by priority. When a grant matches more than one topic:

- all matches are preserved in `matched_topics`;
- `topic_match_count` records the number of matches;
- the highest-priority match becomes `primary_topic`.

The priority system favours narrower and more specific topics before broader umbrella categories. For example, a grant matching both catalysis and general chemistry is assigned `AI-enabled catalysis` as its primary topic.

Grants that do not match one of the eight named topics are retained as `Other AI-enabled chemistry/materials` rather than being forced into a weak or unsupported category.

In [15]:
# Define the shared GrantScopeAI topic taxonomy

topic_rule_rows = [
    {
        "priority": 1,
        "topic": "Autonomous laboratories",
        "keyword_pattern": (
            r"\b(?:"
            r"autonomous laborator(?:y|ies)|"
            r"self[- ]driving laborator(?:y|ies)|"
            r"self[- ]driving lab|"
            r"robotic laborator(?:y|ies)|"
            r"automated experimentation|"
            r"laboratory automation|"
            r"closed[- ]loop experimentation"
            r")\b"
        ),
        "definition": (
            "Automated or closed-loop experimental systems "
            "that plan, execute, and learn from experiments"
        )
    },
    {
        "priority": 2,
        "topic": "Reaction prediction",
        "keyword_pattern": (
            r"\b(?:"
            r"reaction prediction|"
            r"reaction outcome prediction|"
            r"reaction optimization|"
            r"reaction optimisation|"
            r"retrosynthesis|"
            r"retrosynthetic|"
            r"synthesis planning|"
            r"synthetic route planning|"
            r"reaction pathway prediction"
            r")\b"
        ),
        "definition": (
            "Prediction or computational planning of "
            "chemical reactions and synthetic routes"
        )
    },
    {
        "priority": 3,
        "topic": "AI-enabled catalysis",
        "keyword_pattern": (
            r"\b(?:"
            r"catalysis|"
            r"catalyst|"
            r"catalysts|"
            r"catalytic|"
            r"electrocatalysis|"
            r"electrocatalyst|"
            r"photocatalysis|"
            r"photocatalyst|"
            r"biocatalysis|"
            r"biocatalyst"
            r")\b"
        ),
        "definition": (
            "AI- or data-enabled research involving "
            "catalysts and catalytic processes"
        )
    },
    {
        "priority": 4,
        "topic": "Materials informatics",
        "keyword_pattern": (
            r"\b(?:"
            r"materials informatics|"
            r"material informatics|"
            r"materials genome|"
            r"materials database|"
            r"materials data infrastructure|"
            r"materials acceleration platform|"
            r"materials knowledge graph"
            r")\b"
        ),
        "definition": (
            "Structured data, databases, and informatics "
            "methods specifically for materials research"
        )
    },
    {
        "priority": 5,
        "topic": "Cheminformatics",
        "keyword_pattern": (
            r"\b(?:"
            r"cheminformatics|"
            r"chemoinformatics|"
            r"chemical informatics|"
            r"chemical fingerprints?|"
            r"molecular fingerprints?|"
            r"chemical database|"
            r"chemical knowledge graph|"
            r"\bQSAR\b"
            r")"
        ),
        "definition": (
            "Computational representation, storage, and "
            "analysis of chemical and molecular information"
        )
    },
    {
        "priority": 6,
        "topic": "Molecular machine learning",
        "keyword_pattern": (
            r"\b(?:"
            r"molecular machine learning|"
            r"molecular representation learning|"
            r"molecular property prediction|"
            r"molecular graph neural network|"
            r"molecular graph networks?|"
            r"molecular embeddings?|"
            r"molecule generation|"
            r"molecular generation|"
            r"generative molecular design"
            r")\b"
        ),
        "definition": (
            "Machine-learning methods for molecular "
            "representation, generation, or property prediction"
        )
    },
    {
        "priority": 7,
        "topic": "AI-enabled materials",
        "keyword_pattern": (
            r"\b(?:"
            r"materials discovery|"
            r"material discovery|"
            r"materials design|"
            r"material design|"
            r"materials optimization|"
            r"materials optimisation|"
            r"materials property prediction|"
            r"materials simulation|"
            r"advanced materials|"
            r"functional materials|"
            r"battery materials|"
            r"nanomaterials?|"
            r"biomaterials?"
            r")\b"
        ),
        "definition": (
            "AI- or data-enabled discovery, design, "
            "simulation, or optimization of materials"
        )
    },
    {
        "priority": 8,
        "topic": "AI-enabled chemistry",
        "keyword_pattern": (
            r"\b(?:"
            r"computational chemistry|"
            r"quantum chemistry|"
            r"chemical synthesis|"
            r"organic synthesis|"
            r"chemical reactions?|"
            r"chemical processes?|"
            r"electrochemistry|"
            r"electrochemical|"
            r"spectroscopy|"
            r"drug discovery|"
            r"chemistry"
            r")\b"
        ),
        "definition": (
            "General AI- or data-enabled chemistry research "
            "not captured by a more specific topic"
        )
    }
]

topic_mapping_df = (
    pd.DataFrame(topic_rule_rows)
    .sort_values("priority")
    .reset_index(drop=True)
)

print(
    "Topics defined:",
    len(topic_mapping_df)
)

print(
    "Matches OpenAlex topic count:",
    len(topic_mapping_df)
    == openalex_df["topic_clean"].nunique()
)

display(
    topic_mapping_df[
        [
            "priority",
            "topic",
            "definition"
        ]
    ]
)

Topics defined: 8
Matches OpenAlex topic count: True


,priority,topic,definition
0,1,Autonomous laboratories,Automated or closed-loop experimental systems ...
1,2,Reaction prediction,Prediction or computational planning of chemic...
2,3,AI-enabled catalysis,AI- or data-enabled research involving catalys...
3,4,Materials informatics,"Structured data, databases, and informatics me..."
4,5,Cheminformatics,"Computational representation, storage, and ana..."
5,6,Molecular machine learning,Machine-learning methods for molecular represe...
6,7,AI-enabled materials,"AI- or data-enabled discovery, design, simulat..."
7,8,AI-enabled chemistry,General AI- or data-enabled chemistry research...


In [16]:
# Apply the shared topic taxonomy to the integrated grants

import re


# Build the text used for topic assignment
grants_clean_df["topic_search_text"] = (
    grants_clean_df[
        [
            "title",
            "abstract",
            "programme_name",
            "funding_mechanism"
        ]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


def make_topic_match_column(topic_name):
    """Convert a topic label into a safe Boolean-column name."""

    topic_slug = re.sub(
        r"[^a-z0-9]+",
        "_",
        topic_name.lower()
    ).strip("_")

    return f"topic_match_{topic_slug}"


# Create one Boolean match column for each topic
topic_match_columns = {}

for _, rule in topic_mapping_df.iterrows():

    topic_name = rule["topic"]
    keyword_pattern = rule["keyword_pattern"]

    match_column = make_topic_match_column(
        topic_name
    )

    topic_match_columns[topic_name] = match_column

    grants_clean_df[match_column] = (
        grants_clean_df["topic_search_text"]
        .str.contains(
            keyword_pattern,
            case=False,
            na=False,
            regex=True
        )
    )


# Keep topics ordered according to the documented priority
ordered_topics = (
    topic_mapping_df
    .sort_values("priority")["topic"]
    .tolist()
)


def collect_matched_topics(row):
    matched_topics = [
        topic
        for topic in ordered_topics
        if row[topic_match_columns[topic]]
    ]

    return " | ".join(matched_topics)


def assign_primary_topic(row):
    for topic in ordered_topics:
        if row[topic_match_columns[topic]]:
            return topic

    return "Other AI-enabled chemistry/materials"


grants_clean_df["matched_topics"] = (
    grants_clean_df.apply(
        collect_matched_topics,
        axis=1
    )
)

grants_clean_df["topic_match_count"] = (
    grants_clean_df[
        list(topic_match_columns.values())
    ]
    .sum(axis=1)
    .astype("Int64")
)

grants_clean_df["primary_topic"] = (
    grants_clean_df.apply(
        assign_primary_topic,
        axis=1
    )
)


topic_assignment_summary_df = (
    grants_clean_df.groupby(
        ["source", "primary_topic"]
    )
    .size()
    .reset_index(name="grant_count")
    .sort_values(
        ["source", "grant_count"],
        ascending=[True, False]
    )
)


print(
    "Grants assigned to at least one taxonomy topic:",
    f"{grants_clean_df['topic_match_count'].gt(0).sum():,}"
)

print(
    "Grants assigned to multiple topics:",
    f"{grants_clean_df['topic_match_count'].gt(1).sum():,}"
)

print(
    "Grants assigned to the fallback category:",
    f"{grants_clean_df['topic_match_count'].eq(0).sum():,}"
)

print(
    "Maximum topics matched by one grant:",
    grants_clean_df["topic_match_count"].max()
)

print("\nPrimary-topic counts by source:")
display(topic_assignment_summary_df)

Grants assigned to at least one taxonomy topic: 1,836
Grants assigned to multiple topics: 546
Grants assigned to the fallback category: 1,503
Maximum topics matched by one grant: 4

Primary-topic counts by source:


,source,primary_topic,grant_count
6,CORDIS,Other AI-enabled chemistry/materials,403
1,CORDIS,AI-enabled chemistry,135
0,CORDIS,AI-enabled catalysis,70
2,CORDIS,AI-enabled materials,52
4,CORDIS,Cheminformatics,9
7,CORDIS,Reaction prediction,3
5,CORDIS,Materials informatics,2
3,CORDIS,Autonomous laboratories,1
15,NSF,Other AI-enabled chemistry/materials,1100
9,NSF,AI-enabled chemistry,752


In [17]:
# Review grants not assigned to one of the eight named topics

fallback_grants_df = (
    grants_clean_df.loc[
        grants_clean_df["topic_match_count"].eq(0)
    ]
    .copy()
)

fallback_summary_df = (
    fallback_grants_df.groupby(
        ["source", "relevance_tier"]
    )
    .size()
    .reset_index(name="grant_count")
    .sort_values(
        ["source", "relevance_tier"]
    )
)

fallback_review_samples = []

for source_name in ["CORDIS", "NSF"]:

    source_fallback_df = fallback_grants_df.loc[
        fallback_grants_df["source"].eq(source_name)
    ]

    sample_size = min(
        10,
        len(source_fallback_df)
    )

    fallback_review_samples.append(
        source_fallback_df.sample(
            n=sample_size,
            random_state=42
        )
    )

fallback_review_df = pd.concat(
    fallback_review_samples,
    ignore_index=True
)

fallback_review_df["abstract_preview"] = (
    fallback_review_df["abstract"]
    .fillna("")
    .str.slice(0, 350)
)

print(
    "Total fallback grants:",
    f"{len(fallback_grants_df):,}"
)

print("\nFallback counts by source and relevance tier:")
display(fallback_summary_df)

print("\nReproducible fallback sample:")
display(
    fallback_review_df[
        [
            "source",
            "grant_key",
            "award_year",
            "title",
            "programme_name",
            "relevance_tier",
            "abstract_preview"
        ]
    ]
)

Total fallback grants: 1,503

Fallback counts by source and relevance tier:


,source,relevance_tier,grant_count
0,CORDIS,broad_match,231
1,CORDIS,core_match,172
2,NSF,broad_match,735
3,NSF,core_match,365



Reproducible fallback sample:


,source,grant_key,award_year,title,programme_name,relevance_tier,abstract_preview
0,CORDIS,CORDIS_101059988,2023,Marine Protected Areas Europe,Improved science based maritime spatial planni...,broad_match,We will map the optimal locations for Marine P...
1,CORDIS,CORDIS_101169245,2024,PRostate cancer OMics Oriented inTErvention,MSCA Doctoral Networks 2023,broad_match,PROMOTE focuses on multidisciplinary education...
2,CORDIS,CORDIS_101172693,2024,Developing and implementing VIrtual Control gr...,Accelerating the implementation of New Approac...,core_match,"VICT3R, a public-private partnership running u..."
3,CORDIS,CORDIS_101058450,2022,METAL MATRIX NANO-COMPOSITE COATINGS UTILIZATI...,Safe- and sustainable-by-design metallic coati...,core_match,The MOZART project proposes sustainable coatin...
4,CORDIS,CORDIS_101066321,2022,Solid-State Ionics Synaptic Transistors for Ne...,ERC PROOF OF CONCEPT GRANTS1,core_match,Neuromorphic computing will revolutionize arti...
5,CORDIS,CORDIS_101072634,2023,Robotics and Artificial Intelligence for Criti...,MSCA Doctoral Networks 2021,core_match,This project aims to develop technologies whic...
6,CORDIS,CORDIS_101119774,2023,Spatial Perception and Embodied Autonomy Research,Pushing the limit of physical intelligence and...,broad_match,"""The popular unmanned aerial robot designs are..."
7,CORDIS,CORDIS_101225859,2025,Functional cOmposition of post quantum Cryptos...,Post-quantum cryptography transition,broad_match,The FOCAL project will deliver a unified frame...
8,CORDIS,CORDIS_101156161,2025,Noise and/or ultrafine particulate matter indu...,The role of environmental pollution in non-com...,broad_match,"Air pollution, especially particulate matter (..."
9,CORDIS,CORDIS_101187428,2025,INTELLIGENT ENCAPSULATION AND SCREENING PLATFO...,EIC Pathfinder Open,broad_match,"Over the past century, drastic lifestyle chang..."


## Refining topic coverage after manual review

The initial topic rules assigned 1,836 of the 3,339 grants to at least one named topic, leaving 1,503 records in the fallback category.

A reproducible sample of the unmatched grants showed two distinct groups:

- genuinely broad or incidental matches that should remain unclassified;
- relevant projects involving polymers, coatings, composites, semiconductors, additive manufacturing, fracture mechanics, chemical-process engineering, and molecular simulation that were not covered by the initial keyword rules.

Rather than creating additional categories, the two broad umbrella topics are expanded:

- `AI-enabled materials` receives additional terminology for materials classes, manufacturing methods, coatings, composites, semiconductors, and mechanical behaviour;
- `AI-enabled chemistry` receives additional terminology for chemical processes, kinetics, sensing, molecular simulation, and major chemistry subfields.

Terms related only to general robotics, cybersecurity, astronomy, or unrelated biomedical research are not added.

After refinement:

- 403 additional grants receive a named topic;
- named-topic coverage increases to 2,239 grants;
- 1,100 grants remain in `Other AI-enabled chemistry/materials`;
- multi-topic assignments remain permitted.

The remaining fallback records are retained rather than being forced into categories that are not supported by their text.

In [18]:
# Refine the two broad umbrella-topic rules

materials_pattern_refined = (
    r"\b(?:"
    r"materials discovery|"
    r"material discovery|"
    r"materials design|"
    r"material design|"
    r"materials optimization|"
    r"materials optimisation|"
    r"materials property prediction|"
    r"materials simulation|"
    r"advanced materials|"
    r"functional materials|"
    r"battery materials|"
    r"electronic materials|"
    r"photonic materials|"
    r"structural materials|"
    r"two[- ]dimensional materials|"
    r"2d materials|"
    r"nanomaterials?|"
    r"biomaterials?|"
    r"nanocomposites?|"
    r"composite materials?|"
    r"metal matrix composites?|"
    r"coatings?|"
    r"thin films?|"
    r"semiconductors?|"
    r"solid[- ]state materials?|"
    r"alloys?|"
    r"ceramics?|"
    r"metallurgy|"
    r"additive manufacturing|"
    r"mechanics of materials|"
    r"fracture mechanics|"
    r"material fracture|"
    r"material deformation|"
    r"polymers?|"
    r"polymeric"
    r")\b"
)

chemistry_pattern_refined = (
    r"\b(?:"
    r"computational chemistry|"
    r"quantum chemistry|"
    r"chemical synthesis|"
    r"organic synthesis|"
    r"chemical reactions?|"
    r"chemical processes?|"
    r"chemical process control|"
    r"reaction engineering|"
    r"process systems engineering|"
    r"chemical kinetics|"
    r"electrochemistry|"
    r"electrochemical|"
    r"analytical chemistry|"
    r"physical chemistry|"
    r"organic chemistry|"
    r"inorganic chemistry|"
    r"chemical biology|"
    r"chemical analysis|"
    r"chemical sensing|"
    r"molecular sciences?|"
    r"molecular dynamics|"
    r"molecular simulations?|"
    r"molecular spectroscopy|"
    r"spectroscopy|"
    r"drug discovery|"
    r"chemistry"
    r")\b"
)

topic_mapping_df.loc[
    topic_mapping_df["topic"].eq(
        "AI-enabled materials"
    ),
    "keyword_pattern"
] = materials_pattern_refined

topic_mapping_df.loc[
    topic_mapping_df["topic"].eq(
        "AI-enabled chemistry"
    ),
    "keyword_pattern"
] = chemistry_pattern_refined


# Save the current fallback count for comparison
fallback_count_before_refinement = (
    grants_clean_df["topic_match_count"]
    .eq(0)
    .sum()
)


# Reapply every topic rule
for _, rule in topic_mapping_df.iterrows():

    topic_name = rule["topic"]
    keyword_pattern = rule["keyword_pattern"]
    match_column = topic_match_columns[topic_name]

    grants_clean_df[match_column] = (
        grants_clean_df["topic_search_text"]
        .str.contains(
            keyword_pattern,
            case=False,
            na=False,
            regex=True
        )
    )


grants_clean_df["matched_topics"] = (
    grants_clean_df.apply(
        collect_matched_topics,
        axis=1
    )
)

grants_clean_df["topic_match_count"] = (
    grants_clean_df[
        list(topic_match_columns.values())
    ]
    .sum(axis=1)
    .astype("Int64")
)

grants_clean_df["primary_topic"] = (
    grants_clean_df.apply(
        assign_primary_topic,
        axis=1
    )
)


fallback_count_after_refinement = (
    grants_clean_df["topic_match_count"]
    .eq(0)
    .sum()
)

newly_classified_count = (
    fallback_count_before_refinement
    - fallback_count_after_refinement
)


refined_topic_summary_df = (
    grants_clean_df.groupby(
        ["source", "primary_topic"]
    )
    .size()
    .reset_index(name="grant_count")
    .sort_values(
        ["source", "grant_count"],
        ascending=[True, False]
    )
)


print(
    "Fallback grants before refinement:",
    f"{fallback_count_before_refinement:,}"
)

print(
    "Fallback grants after refinement:",
    f"{fallback_count_after_refinement:,}"
)

print(
    "Newly assigned grants:",
    f"{newly_classified_count:,}"
)

print(
    "Grants matching multiple topics:",
    f"{grants_clean_df['topic_match_count'].gt(1).sum():,}"
)

print("\nRefined primary-topic counts:")
display(refined_topic_summary_df)

Fallback grants before refinement: 1,503
Fallback grants after refinement: 1,100
Newly assigned grants: 403
Grants matching multiple topics: 808

Refined primary-topic counts:


,source,primary_topic,grant_count
6,CORDIS,Other AI-enabled chemistry/materials,334
1,CORDIS,AI-enabled chemistry,138
2,CORDIS,AI-enabled materials,118
0,CORDIS,AI-enabled catalysis,70
4,CORDIS,Cheminformatics,9
7,CORDIS,Reaction prediction,3
5,CORDIS,Materials informatics,2
3,CORDIS,Autonomous laboratories,1
10,NSF,AI-enabled materials,795
15,NSF,Other AI-enabled chemistry/materials,766


## Preserving multi-topic grant relationships

Some grants legitimately match more than one research topic. Reducing every record to only its `primary_topic` would lose useful information about interdisciplinary projects.

A separate grant-topic bridge table is therefore created with one row for each valid grant–topic relationship.

This structure:

- preserves all named topic matches;
- supports many-to-many relationships between grants and topics;
- allows topic-level counts and funding summaries to use every relevant classification;
- keeps `primary_topic` available for simpler filtering and display;
- prevents duplicate grant–topic pairs.

Because multi-topic grants appear once under each matched topic, rows and funding values across different topics must not be added together and interpreted as unique-grant totals.

The bridge table contains 3,245 grant-topic relationships representing 2,239 unique grants across all eight named topics.

In [19]:
# Create one row per grant-topic relationship

grant_topic_bridge_parts = []

for topic_name, match_column in topic_match_columns.items():

    topic_matches_df = (
        grants_clean_df.loc[
            grants_clean_df[match_column],
            [
                "grant_key",
                "source",
                "award_year",
                "amount_native",
                "currency",
                "relevance_tier"
            ]
        ]
        .copy()
    )

    topic_matches_df["topic"] = topic_name

    grant_topic_bridge_parts.append(
        topic_matches_df
    )


grant_topic_bridge_df = pd.concat(
    grant_topic_bridge_parts,
    ignore_index=True
)

grant_topic_bridge_df = (
    grant_topic_bridge_df[
        [
            "grant_key",
            "source",
            "award_year",
            "topic",
            "amount_native",
            "currency",
            "relevance_tier"
        ]
    ]
    .sort_values(
        [
            "topic",
            "source",
            "award_year",
            "grant_key"
        ]
    )
    .reset_index(drop=True)
)


duplicate_grant_topic_count = (
    grant_topic_bridge_df[
        ["grant_key", "topic"]
    ]
    .duplicated()
    .sum()
)

bridge_unique_grants = (
    grant_topic_bridge_df["grant_key"]
    .nunique()
)

expected_named_topic_grants = (
    grants_clean_df["topic_match_count"]
    .gt(0)
    .sum()
)


print(
    "Grant-topic relationships:",
    f"{len(grant_topic_bridge_df):,}"
)

print(
    "Unique grants with named topics:",
    f"{bridge_unique_grants:,}"
)

print(
    "Expected grants with named topics:",
    f"{expected_named_topic_grants:,}"
)

print(
    "Grant coverage reconciles:",
    bridge_unique_grants
    == expected_named_topic_grants
)

print(
    "Duplicate grant-topic relationships:",
    duplicate_grant_topic_count
)

print(
    "Unique named topics:",
    grant_topic_bridge_df["topic"].nunique()
)

display(grant_topic_bridge_df.head(10))

Grant-topic relationships: 3,245
Unique grants with named topics: 2,239
Expected grants with named topics: 2,239
Grant coverage reconciles: True
Duplicate grant-topic relationships: 0
Unique named topics: 8


,grant_key,source,award_year,topic,amount_native,currency,relevance_tier
0,CORDIS_101044355,CORDIS,2022,AI-enabled catalysis,2000000.0,EUR,core_match
1,CORDIS_101045008,CORDIS,2022,AI-enabled catalysis,1997993.0,EUR,core_match
2,CORDIS_101046836,CORDIS,2022,AI-enabled catalysis,2871775.0,EUR,core_match
3,CORDIS_101057430,CORDIS,2022,AI-enabled catalysis,6205434.5,EUR,core_match
4,CORDIS_101058593,CORDIS,2022,AI-enabled catalysis,4997125.0,EUR,core_match
5,CORDIS_101058643,CORDIS,2022,AI-enabled catalysis,5707393.0,EUR,core_match
6,CORDIS_101058756,CORDIS,2022,AI-enabled catalysis,8358044.0,EUR,core_match
7,CORDIS_101062692,CORDIS,2022,AI-enabled catalysis,290444.16,EUR,core_match
8,CORDIS_101063836,CORDIS,2022,AI-enabled catalysis,214934.4,EUR,core_match
9,CORDIS_101064374,CORDIS,2022,AI-enabled catalysis,183530.4,EUR,core_match


## Creating topic-year funding summaries

The grant-topic bridge is aggregated to create annual funding summaries for each combination of:

- funding source;
- native currency;
- research topic;
- award year.

For each combination, the summary records:

- unique grant count;
- core-match count;
- broad-match count;
- total native funding;
- median native award amount;
- mean native award amount.

Funding remains separated by source and currency. CORDIS values remain in EUR and NSF values remain in USD. No combined cross-currency funding total is created.

Because grants may match multiple topics, topic-level grant counts and funding totals overlap. These values describe activity within each topic and must not be added across topics as though they represent distinct grants.

The initial aggregation contains only topic-year combinations with observed grants. A complete grid is created in the following step so that missing combinations are represented explicitly as zero-grant rows.

In [20]:
# Create funding summaries by source, currency, topic, and year

grant_topic_bridge_df["is_core_match"] = (
    grant_topic_bridge_df["relevance_tier"]
    .eq("core_match")
)

grant_topic_bridge_df["is_broad_match"] = (
    grant_topic_bridge_df["relevance_tier"]
    .eq("broad_match")
)


grant_topic_year_summary_df = (
    grant_topic_bridge_df
    .groupby(
        [
            "source",
            "currency",
            "topic",
            "award_year"
        ],
        dropna=False
    )
    .agg(
        grant_count=(
            "grant_key",
            "nunique"
        ),
        core_match_count=(
            "is_core_match",
            "sum"
        ),
        broad_match_count=(
            "is_broad_match",
            "sum"
        ),
        total_amount_native=(
            "amount_native",
            "sum"
        ),
        median_amount_native=(
            "amount_native",
            "median"
        ),
        mean_amount_native=(
            "amount_native",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        [
            "topic",
            "source",
            "award_year"
        ]
    )
    .reset_index(drop=True)
)


grant_topic_year_summary_df[
    "core_match_count"
] = (
    grant_topic_year_summary_df[
        "core_match_count"
    ]
    .astype("Int64")
)

grant_topic_year_summary_df[
    "broad_match_count"
] = (
    grant_topic_year_summary_df[
        "broad_match_count"
    ]
    .astype("Int64")
)

grant_topic_year_summary_df[
    "grant_count"
] = (
    grant_topic_year_summary_df[
        "grant_count"
    ]
    .astype("Int64")
)


summary_count_reconciliation = (
    grant_topic_year_summary_df[
        "core_match_count"
    ]
    + grant_topic_year_summary_df[
        "broad_match_count"
    ]
).eq(
    grant_topic_year_summary_df[
        "grant_count"
    ]
).all()


print(
    "Topic-year summary rows:",
    f"{len(grant_topic_year_summary_df):,}"
)

print(
    "Topics represented:",
    grant_topic_year_summary_df[
        "topic"
    ].nunique()
)

print(
    "Years represented:",
    sorted(
        grant_topic_year_summary_df[
            "award_year"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
)

print(
    "Sources represented:",
    grant_topic_year_summary_df[
        "source"
    ].unique().tolist()
)

print(
    "Currencies represented:",
    grant_topic_year_summary_df[
        "currency"
    ].unique().tolist()
)

print(
    "Core and broad counts reconcile:",
    summary_count_reconciliation
)

display(grant_topic_year_summary_df.head(15))

Topic-year summary rows: 61
Topics represented: 8
Years represented: [2021, 2022, 2023, 2024, 2025]
Sources represented: ['CORDIS', 'NSF']
Currencies represented: ['EUR', 'USD']
Core and broad counts reconcile: True


,source,currency,topic,award_year,grant_count,core_match_count,broad_match_count,total_amount_native,median_amount_native,mean_amount_native
0,CORDIS,EUR,AI-enabled catalysis,2022,15,15,0,39683004.53,2000000.0,2645533.635333
1,CORDIS,EUR,AI-enabled catalysis,2023,16,16,0,56207132.83,2247750.0,3512945.801875
2,CORDIS,EUR,AI-enabled catalysis,2024,20,20,0,83443512.76,2499762.5,4172175.638
3,CORDIS,EUR,AI-enabled catalysis,2025,21,19,2,52583287.39,1998106.0,2503966.06619
4,NSF,USD,AI-enabled catalysis,2021,40,38,2,81498247.0,498174.0,2037456.175
5,NSF,USD,AI-enabled catalysis,2022,53,50,3,40489403.0,464139.0,763951.0
6,NSF,USD,AI-enabled catalysis,2023,66,55,11,256721717.0,526517.0,3889722.984848
7,NSF,USD,AI-enabled catalysis,2024,53,53,0,139491363.0,575000.0,2631912.509434
8,NSF,USD,AI-enabled catalysis,2025,68,64,4,81151350.0,500322.5,1193402.205882
9,CORDIS,EUR,AI-enabled chemistry,2022,38,36,2,120218134.78,2438370.94,3163635.125789


## Completing the topic-year funding grid

The initial topic-year summary includes only combinations where at least one grant was observed.

For comparison, visualization, and later integration with OpenAlex, missing topic-year combinations should be represented explicitly rather than disappearing from the dataset.

A complete grid is therefore created containing:

- eight research topics;
- five years from 2021 through 2025;
- two funding sources;
- the correct native currency for each source.

This produces 80 expected combinations.

Where no grants exist:

- grant, core-match, and broad-match counts are set to zero;
- total native funding is set to zero;
- mean and median award values remain missing because no award distribution exists.

This distinction prevents an absent combination from being confused with missing data while avoiding misleading average values for groups with no grants.

In [21]:
# Complete the funding topic-year grid with explicit zero-grant rows

all_topics = (
    topic_mapping_df
    .sort_values("priority")["topic"]
    .tolist()
)

all_years = [
    2021,
    2022,
    2023,
    2024,
    2025
]

source_currency_map = {
    "CORDIS": "EUR",
    "NSF": "USD"
}


complete_topic_year_rows = []

for source_name, currency_code in source_currency_map.items():

    for topic_name in all_topics:

        for year in all_years:

            complete_topic_year_rows.append({
                "source": source_name,
                "currency": currency_code,
                "topic": topic_name,
                "award_year": year
            })


complete_topic_year_grid_df = pd.DataFrame(
    complete_topic_year_rows
)


grant_topic_year_complete_df = (
    complete_topic_year_grid_df
    .merge(
        grant_topic_year_summary_df,
        on=[
            "source",
            "currency",
            "topic",
            "award_year"
        ],
        how="left"
    )
)


count_columns = [
    "grant_count",
    "core_match_count",
    "broad_match_count"
]

amount_columns = [
    "total_amount_native",
    "median_amount_native",
    "mean_amount_native"
]


for column in count_columns:

    grant_topic_year_complete_df[column] = (
        grant_topic_year_complete_df[column]
        .fillna(0)
        .astype("Int64")
    )


# A missing total means no grants existed in that combination
grant_topic_year_complete_df[
    "total_amount_native"
] = (
    grant_topic_year_complete_df[
        "total_amount_native"
    ]
    .fillna(0)
    .astype("Float64")
)

# Median and mean remain missing where no grants exist
for column in [
    "median_amount_native",
    "mean_amount_native"
]:

    grant_topic_year_complete_df[column] = (
        grant_topic_year_complete_df[column]
        .astype("Float64")
    )


grant_topic_year_complete_df = (
    grant_topic_year_complete_df
    .sort_values(
        [
            "topic",
            "source",
            "award_year"
        ]
    )
    .reset_index(drop=True)
)


zero_grant_rows = (
    grant_topic_year_complete_df[
        "grant_count"
    ]
    .eq(0)
    .sum()
)


print(
    "Complete topic-year rows:",
    len(grant_topic_year_complete_df)
)

print(
    "Expected rows:",
    len(all_topics)
    * len(all_years)
    * len(source_currency_map)
)

print(
    "Rows representing zero grants:",
    zero_grant_rows
)

print(
    "Missing total funding values:",
    grant_topic_year_complete_df[
        "total_amount_native"
    ]
    .isna()
    .sum()
)

print(
    "Core and broad counts reconcile:",
    (
        grant_topic_year_complete_df[
            "core_match_count"
        ]
        + grant_topic_year_complete_df[
            "broad_match_count"
        ]
    )
    .eq(
        grant_topic_year_complete_df[
            "grant_count"
        ]
    )
    .all()
)

display(
    grant_topic_year_complete_df.head(15)
)

Complete topic-year rows: 80
Expected rows: 80
Rows representing zero grants: 19
Missing total funding values: 0
Core and broad counts reconcile: True


,source,currency,topic,award_year,grant_count,core_match_count,broad_match_count,total_amount_native,median_amount_native,mean_amount_native
0,CORDIS,EUR,AI-enabled catalysis,2021,0,0,0,0.0,<NA>,<NA>
1,CORDIS,EUR,AI-enabled catalysis,2022,15,15,0,39683004.53,2000000.0,2645533.635333
2,CORDIS,EUR,AI-enabled catalysis,2023,16,16,0,56207132.83,2247750.0,3512945.801875
3,CORDIS,EUR,AI-enabled catalysis,2024,20,20,0,83443512.76,2499762.5,4172175.638
4,CORDIS,EUR,AI-enabled catalysis,2025,21,19,2,52583287.39,1998106.0,2503966.06619
5,NSF,USD,AI-enabled catalysis,2021,40,38,2,81498247.0,498174.0,2037456.175
6,NSF,USD,AI-enabled catalysis,2022,53,50,3,40489403.0,464139.0,763951.0
7,NSF,USD,AI-enabled catalysis,2023,66,55,11,256721717.0,526517.0,3889722.984848
8,NSF,USD,AI-enabled catalysis,2024,53,53,0,139491363.0,575000.0,2631912.509434
9,NSF,USD,AI-enabled catalysis,2025,68,64,4,81151350.0,500322.5,1193402.205882


## Aligning funding topics with OpenAlex

OpenAlex remains a separate publication-context dataset, but its topic labels and years must align exactly with the shared funding taxonomy before comparison.

The alignment check confirms that:

- OpenAlex contains the same eight named topics used for NSF and CORDIS;
- topic labels match exactly after standardization;
- publication years fall within the shared 2021–2025 period;
- every topic-year combination is unique;
- no expected taxonomy topics are missing;
- no unexpected OpenAlex topics are present.

This step validates compatibility at the topic-year level without merging publication records directly into individual grants.

OpenAlex publication counts provide research-activity context only. They do not represent grant records and are not used to calculate grant-level funding values.

In [22]:
# Validate OpenAlex alignment with the shared topic taxonomy

openalex_context_df = openalex_df.copy()

openalex_context_df["topic"] = (
    openalex_context_df["topic_clean"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

openalex_context_df["year"] = (
    pd.to_numeric(
        openalex_context_df["publication_year"],
        errors="coerce"
    )
    .astype("Int64")
)

openalex_context_df["publication_count"] = (
    pd.to_numeric(
        openalex_context_df["publication_count"],
        errors="coerce"
    )
    .astype("Int64")
)


taxonomy_topic_set = set(all_topics)

openalex_topic_set = set(
    openalex_context_df["topic"]
    .dropna()
    .tolist()
)

topics_missing_from_openalex = sorted(
    taxonomy_topic_set
    - openalex_topic_set
)

unexpected_openalex_topics = sorted(
    openalex_topic_set
    - taxonomy_topic_set
)

openalex_duplicate_topic_years = (
    openalex_context_df[
        ["topic", "year"]
    ]
    .duplicated()
    .sum()
)

invalid_openalex_years = (
    ~openalex_context_df["year"]
    .isin(all_years)
).sum()


print(
    "OpenAlex rows:",
    len(openalex_context_df)
)

print(
    "Unique OpenAlex topics:",
    openalex_context_df["topic"].nunique()
)

print(
    "Duplicate OpenAlex topic-year rows:",
    openalex_duplicate_topic_years
)

print(
    "Invalid OpenAlex years:",
    invalid_openalex_years
)

print(
    "Taxonomy topics missing from OpenAlex:",
    topics_missing_from_openalex
)

print(
    "Unexpected OpenAlex topics:",
    unexpected_openalex_topics
)

print(
    "Topic sets align exactly:",
    taxonomy_topic_set
    == openalex_topic_set
)

OpenAlex rows: 40
Unique OpenAlex topics: 8
Duplicate OpenAlex topic-year rows: 0
Invalid OpenAlex years: 0
Taxonomy topics missing from OpenAlex: []
Unexpected OpenAlex topics: []
Topic sets align exactly: True


## Creating the three-source topic-year context table

The completed funding summaries are combined with the OpenAlex publication dataset at the shared topic-year level.

This table does not merge publications with individual grants. Instead, it places three complementary indicators side by side for each research topic and year:

- CORDIS grant counts and funding values;
- NSF grant counts and funding values;
- OpenAlex publication counts and publication-growth measures.

CORDIS and NSF funding remain in separate columns because they use different native currencies:

- CORDIS funding is reported in EUR;
- NSF funding is reported in USD.

No cross-currency total is calculated.

The resulting table contains one row for each of the eight topics in each year from 2021 through 2025, producing 40 unique topic-year records.

This structure supports later exploratory analysis and dashboarding while preserving the different analytical meanings of funding awards and publication activity.

In [23]:
# Combine funding summaries with OpenAlex publication context

cordis_topic_year_df = (
    grant_topic_year_complete_df.loc[
        grant_topic_year_complete_df["source"].eq("CORDIS"),
        [
            "topic",
            "award_year",
            "grant_count",
            "core_match_count",
            "broad_match_count",
            "total_amount_native",
            "median_amount_native",
            "mean_amount_native"
        ]
    ]
    .rename(
        columns={
            "award_year": "year",
            "grant_count": "cordis_grant_count",
            "core_match_count": "cordis_core_match_count",
            "broad_match_count": "cordis_broad_match_count",
            "total_amount_native": "cordis_total_amount_eur",
            "median_amount_native": "cordis_median_amount_eur",
            "mean_amount_native": "cordis_mean_amount_eur"
        }
    )
)

nsf_topic_year_df = (
    grant_topic_year_complete_df.loc[
        grant_topic_year_complete_df["source"].eq("NSF"),
        [
            "topic",
            "award_year",
            "grant_count",
            "core_match_count",
            "broad_match_count",
            "total_amount_native",
            "median_amount_native",
            "mean_amount_native"
        ]
    ]
    .rename(
        columns={
            "award_year": "year",
            "grant_count": "nsf_grant_count",
            "core_match_count": "nsf_core_match_count",
            "broad_match_count": "nsf_broad_match_count",
            "total_amount_native": "nsf_total_amount_usd",
            "median_amount_native": "nsf_median_amount_usd",
            "mean_amount_native": "nsf_mean_amount_usd"
        }
    )
)


openalex_comparison_columns = [
    "topic",
    "year",
    "publication_count",
    "publication_change",
    "publication_growth_yoy_pct",
    "publication_index_2021"
]

topic_year_context_df = (
    openalex_context_df[
        openalex_comparison_columns
    ]
    .merge(
        cordis_topic_year_df,
        on=["topic", "year"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        nsf_topic_year_df,
        on=["topic", "year"],
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        ["topic", "year"]
    )
    .reset_index(drop=True)
)


grant_count_columns = [
    "cordis_grant_count",
    "cordis_core_match_count",
    "cordis_broad_match_count",
    "nsf_grant_count",
    "nsf_core_match_count",
    "nsf_broad_match_count"
]

missing_grant_counts = (
    topic_year_context_df[
        grant_count_columns
    ]
    .isna()
    .sum()
    .sum()
)

duplicate_topic_years = (
    topic_year_context_df[
        ["topic", "year"]
    ]
    .duplicated()
    .sum()
)


print(
    "Topic-year context rows:",
    len(topic_year_context_df)
)

print(
    "Unique topics:",
    topic_year_context_df["topic"].nunique()
)

print(
    "Years:",
    sorted(
        topic_year_context_df["year"]
        .astype(int)
        .unique()
        .tolist()
    )
)

print(
    "Duplicate topic-year combinations:",
    duplicate_topic_years
)

print(
    "Missing funding-count values:",
    missing_grant_counts
)

display(topic_year_context_df.head(10))

Topic-year context rows: 40
Unique topics: 8
Years: [2021, 2022, 2023, 2024, 2025]
Duplicate topic-year combinations: 0
Missing funding-count values: 0


,topic,year,publication_count,publication_change,publication_growth_yoy_pct,publication_index_2021,cordis_grant_count,cordis_core_match_count,cordis_broad_match_count,cordis_total_amount_eur,cordis_median_amount_eur,cordis_mean_amount_eur,nsf_grant_count,nsf_core_match_count,nsf_broad_match_count,nsf_total_amount_usd,nsf_median_amount_usd,nsf_mean_amount_usd
0,AI-enabled catalysis,2021,6094,NaN,NaN,100.00,0,0,0,0.0,<NA>,<NA>,40,38,2,81498247.0,498174.0,2037456.175
1,AI-enabled catalysis,2022,8449,2355.0,38.64,138.64,15,15,0,39683004.53,2000000.0,2645533.635333,53,50,3,40489403.0,464139.0,763951.0
2,AI-enabled catalysis,2023,18137,9688.0,114.66,297.62,16,16,0,56207132.83,2247750.0,3512945.801875,66,55,11,256721717.0,526517.0,3889722.984848
3,AI-enabled catalysis,2024,25744,7607.0,41.94,422.45,20,20,0,83443512.76,2499762.5,4172175.638,53,53,0,139491363.0,575000.0,2631912.509434
4,AI-enabled catalysis,2025,39893,14149.0,54.96,654.63,21,19,2,52583287.39,1998106.0,2503966.06619,68,64,4,81151350.0,500322.5,1193402.205882
5,AI-enabled chemistry,2021,40651,NaN,NaN,100.00,0,0,0,0.0,<NA>,<NA>,201,179,22,334716409.0,480000.0,1665255.766169
6,AI-enabled chemistry,2022,54337,13686.0,33.67,133.67,38,36,2,120218134.78,2438370.94,3163635.125789,243,226,17,208721218.0,450000.0,858935.053498
7,AI-enabled chemistry,2023,96605,42268.0,77.79,237.64,60,51,9,143344008.12,2031385.6,2389066.802,272,239,33,656974490.0,490893.0,2415347.389706
8,AI-enabled chemistry,2024,112587,15982.0,16.54,276.96,62,59,3,204219836.49,2627393.6,3293868.330484,257,225,32,308397316.0,456613.0,1199989.55642
9,AI-enabled chemistry,2025,142246,29659.0,26.34,349.92,68,63,5,197999448.33,1491302.0,2911756.593088,326,282,44,319792131.0,461060.0,980957.457055


## Building the integrated data-quality report

A common machine-readable quality report is created to consolidate validation across NSF, CORDIS, the grant-topic bridge, OpenAlex, and the final topic-year context table.

Each quality rule records:

- the source or table being checked;
- the processing stage;
- the validation rule;
- the number of records checked;
- the number and rate of failures;
- the severity of the issue;
- the required action;
- example record identifiers where relevant.

The report includes checks for:

- source row-count reconciliation;
- globally unique grant keys;
- missing titles, abstracts, organisations, amounts, and URLs;
- invalid dates, currencies, relevance tiers, or negative values;
- duplicate grant-topic relationships;
- OpenAlex topic-year completeness;
- alignment between the funding taxonomy and OpenAlex;
- uniqueness of the final 40-row topic-year context table.

Most rules represent strict data-quality requirements and should return `PASS`.

The topic-assignment coverage rule is intentionally informational. Grants without a named taxonomy match remain valid candidate records and are retained under `Other AI-enabled chemistry/materials`. Therefore, this rule may return `REVIEW` without indicating a failed integration process.

In [24]:
# Build the integrated data-quality report

def get_example_ids(dataframe, mask, id_column, limit=5):
    """Return a short pipe-separated sample of affected record IDs."""

    return " | ".join(
        dataframe.loc[
            mask,
            id_column
        ]
        .dropna()
        .astype(str)
        .head(limit)
        .tolist()
    )


quality_report_rows = []


def add_quality_rule(
    source,
    stage,
    rule,
    records_checked,
    failures,
    severity,
    action,
    example_ids=""
):
    failure_rate = (
        failures / records_checked
        if records_checked
        else 0
    )

    quality_report_rows.append({
        "source": source,
        "stage": stage,
        "rule": rule,
        "records_checked": records_checked,
        "failures": failures,
        "failure_rate": round(
            failure_rate,
            4
        ),
        "severity": severity,
        "action": action,
        "example_ids": example_ids
    })


# --------------------------------------------------
# Integrated grant-table rules
# --------------------------------------------------

duplicate_grant_mask = (
    grants_clean_df["grant_key"]
    .duplicated(keep=False)
)

missing_title_mask = (
    grants_clean_df["title"]
    .isna()
)

missing_abstract_mask = (
    grants_clean_df["abstract"]
    .isna()
)

missing_amount_mask = (
    grants_clean_df["amount_native"]
    .isna()
)

negative_amount_mask = (
    grants_clean_df["amount_native"]
    .lt(0)
    .fillna(False)
)

missing_organisation_mask = (
    grants_clean_df["organisation_name"]
    .isna()
)

invalid_url_mask = (
    ~grants_clean_df["source_url"]
    .fillna("")
    .str.startswith(
        ("http://", "https://")
    )
)

outside_scope_mask = (
    ~grants_clean_df["award_year"]
    .between(2021, 2025)
    .fillna(False)
)

currency_mismatch_mask = (
    (
        grants_clean_df["source"].eq("CORDIS")
        & grants_clean_df["currency"].ne("EUR")
    )
    |
    (
        grants_clean_df["source"].eq("NSF")
        & grants_clean_df["currency"].ne("USD")
    )
)

unexpected_relevance_mask = (
    ~grants_clean_df["relevance_tier"]
    .isin([
        "core_match",
        "broad_match"
    ])
)

fallback_topic_mask = (
    grants_clean_df["topic_match_count"]
    .eq(0)
)


add_quality_rule(
    source="ALL_FUNDING",
    stage="integration",
    rule="Combined row count reconciles with source inputs",
    records_checked=1,
    failures=int(
        len(grants_clean_df)
        != (
            len(cordis_standard_df)
            + len(nsf_standard_df)
        )
    ),
    severity="critical",
    action="Correct integration logic if row counts do not reconcile"
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="integration",
    rule="Grant keys are globally unique",
    records_checked=len(grants_clean_df),
    failures=int(
        duplicate_grant_mask.sum()
    ),
    severity="critical",
    action="Investigate and resolve duplicate grant keys",
    example_ids=get_example_ids(
        grants_clean_df,
        duplicate_grant_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="completeness",
    rule="Titles are present",
    records_checked=len(grants_clean_df),
    failures=int(
        missing_title_mask.sum()
    ),
    severity="high",
    action="Exclude or recover records with missing titles",
    example_ids=get_example_ids(
        grants_clean_df,
        missing_title_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="completeness",
    rule="Abstracts are present",
    records_checked=len(grants_clean_df),
    failures=int(
        missing_abstract_mask.sum()
    ),
    severity="high",
    action="Exclude from text modelling if abstract is unavailable",
    example_ids=get_example_ids(
        grants_clean_df,
        missing_abstract_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="funding",
    rule="Primary funding amounts are present",
    records_checked=len(grants_clean_df),
    failures=int(
        missing_amount_mask.sum()
    ),
    severity="high",
    action="Flag records without a usable primary amount",
    example_ids=get_example_ids(
        grants_clean_df,
        missing_amount_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="funding",
    rule="Primary funding amounts are non-negative",
    records_checked=len(grants_clean_df),
    failures=int(
        negative_amount_mask.sum()
    ),
    severity="high",
    action="Investigate negative funding values",
    example_ids=get_example_ids(
        grants_clean_df,
        negative_amount_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="completeness",
    rule="Lead organisations are present",
    records_checked=len(grants_clean_df),
    failures=int(
        missing_organisation_mask.sum()
    ),
    severity="medium",
    action="Flag records without an identifiable lead organisation",
    example_ids=get_example_ids(
        grants_clean_df,
        missing_organisation_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="traceability",
    rule="Source URLs are valid HTTP links",
    records_checked=len(grants_clean_df),
    failures=int(
        invalid_url_mask.sum()
    ),
    severity="high",
    action="Repair or flag invalid source links",
    example_ids=get_example_ids(
        grants_clean_df,
        invalid_url_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="scope",
    rule="Award years fall within 2021–2025",
    records_checked=len(grants_clean_df),
    failures=int(
        outside_scope_mask.sum()
    ),
    severity="high",
    action="Exclude records outside the documented project period",
    example_ids=get_example_ids(
        grants_clean_df,
        outside_scope_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="funding",
    rule="Currency matches funding source",
    records_checked=len(grants_clean_df),
    failures=int(
        currency_mismatch_mask.sum()
    ),
    severity="critical",
    action="Correct source-currency mismatches",
    example_ids=get_example_ids(
        grants_clean_df,
        currency_mismatch_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="classification",
    rule="Relevance tiers use approved values",
    records_checked=len(grants_clean_df),
    failures=int(
        unexpected_relevance_mask.sum()
    ),
    severity="high",
    action="Correct unexpected relevance labels",
    example_ids=get_example_ids(
        grants_clean_df,
        unexpected_relevance_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="classification",
    rule="Grant receives at least one named taxonomy topic",
    records_checked=len(grants_clean_df),
    failures=int(
        fallback_topic_mask.sum()
    ),
    severity="informational",
    action=(
        "Retain unmatched candidates in "
        "'Other AI-enabled chemistry/materials'"
    ),
    example_ids=get_example_ids(
        grants_clean_df,
        fallback_topic_mask,
        "grant_key"
    )
)


# --------------------------------------------------
# Grant-topic bridge rules
# --------------------------------------------------

duplicate_bridge_mask = (
    grant_topic_bridge_df[
        ["grant_key", "topic"]
    ]
    .duplicated(keep=False)
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="topic_bridge",
    rule="Grant-topic relationships are unique",
    records_checked=len(grant_topic_bridge_df),
    failures=int(
        duplicate_bridge_mask.sum()
    ),
    severity="critical",
    action="Remove duplicate grant-topic relationships",
    example_ids=get_example_ids(
        grant_topic_bridge_df,
        duplicate_bridge_mask,
        "grant_key"
    )
)

add_quality_rule(
    source="ALL_FUNDING",
    stage="topic_bridge",
    rule="Bridge grant coverage reconciles with named-topic grants",
    records_checked=1,
    failures=int(
        grant_topic_bridge_df[
            "grant_key"
        ].nunique()
        != grants_clean_df[
            "topic_match_count"
        ].gt(0).sum()
    ),
    severity="critical",
    action="Correct topic-bridge construction"
)


# --------------------------------------------------
# OpenAlex rules
# --------------------------------------------------

openalex_duplicate_mask = (
    openalex_context_df[
        ["topic", "year"]
    ]
    .duplicated(keep=False)
)

openalex_missing_count_mask = (
    openalex_context_df[
        "publication_count"
    ]
    .isna()
)

openalex_negative_count_mask = (
    openalex_context_df[
        "publication_count"
    ]
    .lt(0)
    .fillna(False)
)

openalex_invalid_year_mask = (
    ~openalex_context_df[
        "year"
    ]
    .isin(all_years)
)


add_quality_rule(
    source="OpenAlex",
    stage="publication_context",
    rule="Topic-year combinations are unique",
    records_checked=len(openalex_context_df),
    failures=int(
        openalex_duplicate_mask.sum()
    ),
    severity="critical",
    action="Resolve duplicate topic-year records",
    example_ids=get_example_ids(
        openalex_context_df,
        openalex_duplicate_mask,
        "topic_year_key"
    )
)

add_quality_rule(
    source="OpenAlex",
    stage="publication_context",
    rule="Publication counts are present",
    records_checked=len(openalex_context_df),
    failures=int(
        openalex_missing_count_mask.sum()
    ),
    severity="high",
    action="Recover or flag missing publication counts",
    example_ids=get_example_ids(
        openalex_context_df,
        openalex_missing_count_mask,
        "topic_year_key"
    )
)

add_quality_rule(
    source="OpenAlex",
    stage="publication_context",
    rule="Publication counts are non-negative",
    records_checked=len(openalex_context_df),
    failures=int(
        openalex_negative_count_mask.sum()
    ),
    severity="high",
    action="Investigate invalid publication counts",
    example_ids=get_example_ids(
        openalex_context_df,
        openalex_negative_count_mask,
        "topic_year_key"
    )
)

add_quality_rule(
    source="OpenAlex",
    stage="publication_context",
    rule="Publication years fall within 2021–2025",
    records_checked=len(openalex_context_df),
    failures=int(
        openalex_invalid_year_mask.sum()
    ),
    severity="high",
    action="Exclude records outside the project period",
    example_ids=get_example_ids(
        openalex_context_df,
        openalex_invalid_year_mask,
        "topic_year_key"
    )
)

add_quality_rule(
    source="OpenAlex",
    stage="taxonomy_alignment",
    rule="OpenAlex and funding taxonomy topic sets align",
    records_checked=1,
    failures=int(
        taxonomy_topic_set
        != openalex_topic_set
    ),
    severity="critical",
    action="Reconcile topic labels across sources"
)

add_quality_rule(
    source="ALL_SOURCES",
    stage="topic_year_context",
    rule="Integrated topic-year context contains 40 unique rows",
    records_checked=1,
    failures=int(
        len(topic_year_context_df) != 40
        or topic_year_context_df[
            ["topic", "year"]
        ].duplicated().any()
    ),
    severity="critical",
    action="Correct topic-year integration logic"
)


data_quality_report_df = pd.DataFrame(
    quality_report_rows
)

data_quality_report_df["status"] = (
    data_quality_report_df["failures"]
    .eq(0)
    .map({
        True: "PASS",
        False: "REVIEW"
    })
)

data_quality_report_df = (
    data_quality_report_df[
        [
            "source",
            "stage",
            "rule",
            "records_checked",
            "failures",
            "failure_rate",
            "severity",
            "status",
            "action",
            "example_ids"
        ]
    ]
)


print(
    "Quality rules:",
    len(data_quality_report_df)
)

print(
    "Passed rules:",
    data_quality_report_df[
        "status"
    ].eq("PASS").sum()
)

print(
    "Rules requiring review:",
    data_quality_report_df[
        "status"
    ].eq("REVIEW").sum()
)

display(data_quality_report_df)

Quality rules: 20
Passed rules: 19
Rules requiring review: 1


,source,stage,rule,records_checked,failures,failure_rate,severity,status,action,example_ids
0,ALL_FUNDING,integration,Combined row count reconciles with source inputs,1,0,0.0000,critical,PASS,Correct integration logic if row counts do not...,
1,ALL_FUNDING,integration,Grant keys are globally unique,3339,0,0.0000,critical,PASS,Investigate and resolve duplicate grant keys,
2,ALL_FUNDING,completeness,Titles are present,3339,0,0.0000,high,PASS,Exclude or recover records with missing titles,
3,ALL_FUNDING,completeness,Abstracts are present,3339,0,0.0000,high,PASS,Exclude from text modelling if abstract is una...,
4,ALL_FUNDING,funding,Primary funding amounts are present,3339,0,0.0000,high,PASS,Flag records without a usable primary amount,
5,ALL_FUNDING,funding,Primary funding amounts are non-negative,3339,0,0.0000,high,PASS,Investigate negative funding values,
6,ALL_FUNDING,completeness,Lead organisations are present,3339,0,0.0000,medium,PASS,Flag records without an identifiable lead orga...,
7,ALL_FUNDING,traceability,Source URLs are valid HTTP links,3339,0,0.0000,high,PASS,Repair or flag invalid source links,
8,ALL_FUNDING,scope,Award years fall within 2021–2025,3339,0,0.0000,high,PASS,Exclude records outside the documented project...,
9,ALL_FUNDING,funding,Currency matches funding source,3339,0,0.0000,critical,PASS,Correct source-currency mismatches,


## Preparing the final integrated grant table

The working integration table contains temporary fields used for validation and topic assignment, including combined search text and individual Boolean topic-match columns.

These helper fields are useful during processing but are not required in the final analytical dataset.

The export-ready grant table therefore retains:

- source and identifier fields;
- titles and abstracts;
- standardized dates and award years;
- native funding amounts and currencies;
- organisation and programme information;
- status and activity classifications;
- source URLs and relevance tiers;
- primary and multi-topic assignments.

Internal regex-match fields and temporary search text are excluded to keep the final table compact and easier to use in Python, Tableau, and Streamlit.

The final table contains:

- 3,339 unique funding records;
- 675 CORDIS projects;
- 2,664 NSF awards;
- 25 application-relevant columns;
- no duplicate grant keys;
- a primary topic for every record, including the documented fallback category.

In [25]:
# Create the final export-ready integrated grant table

final_grant_columns = [
    "source",
    "grant_key",
    "source_id",
    "title",
    "abstract",
    "start_date",
    "end_date",
    "award_year",
    "amount_native",
    "project_total_cost",
    "amount_obligated",
    "currency",
    "organisation_name",
    "organisation_city",
    "organisation_country",
    "programme_name",
    "funding_mechanism",
    "status",
    "activity_type",
    "source_activity_type",
    "source_url",
    "relevance_tier",
    "primary_topic",
    "matched_topics",
    "topic_match_count"
]


grants_export_df = (
    grants_clean_df[
        final_grant_columns
    ]
    .copy()
    .sort_values(
        [
            "source",
            "award_year",
            "grant_key"
        ]
    )
    .reset_index(drop=True)
)


export_source_counts_df = (
    grants_export_df["source"]
    .value_counts()
    .rename_axis("source")
    .reset_index(name="grant_count")
    .sort_values("source")
)


print(
    "Final grant rows:",
    f"{len(grants_export_df):,}"
)

print(
    "Final grant columns:",
    len(grants_export_df.columns)
)

print(
    "Duplicate grant keys:",
    grants_export_df["grant_key"]
    .duplicated()
    .sum()
)

print(
    "Missing primary topics:",
    grants_export_df["primary_topic"]
    .isna()
    .sum()
)

print(
    "Fallback-topic grants:",
    grants_export_df["topic_match_count"]
    .eq(0)
    .sum()
)

print(
    "Named-topic grants:",
    grants_export_df["topic_match_count"]
    .gt(0)
    .sum()
)

print("\nRows by source:")
display(export_source_counts_df)

display(grants_export_df.head())

Final grant rows: 3,339
Final grant columns: 25
Duplicate grant keys: 0
Missing primary topics: 0
Fallback-topic grants: 1100
Named-topic grants: 2239

Rows by source:


,source,grant_count
1,CORDIS,675
0,NSF,2664


,source,grant_key,source_id,title,abstract,start_date,end_date,award_year,amount_native,project_total_cost,...,programme_name,funding_mechanism,status,activity_type,source_activity_type,source_url,relevance_tier,primary_topic,matched_topics,topic_match_count
0,CORDIS,CORDIS_101039636,101039636,"Piezoelectric Biomolecules for lead-free, Reli...",Billions of piezoelectric sensors are produced...,2022-06-01,2027-05-31,2022,1499525.0,1499525.0,...,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101039636,core_match,Other AI-enabled chemistry/materials,,0
1,CORDIS,CORDIS_101040353,101040353,Exploring the Molecular Properties of Atmosphe...,Aerosol particles are ubiquitous constituents ...,2022-04-01,2027-03-31,2022,1462491.0,1462491.0,...,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101040353,core_match,AI-enabled chemistry,AI-enabled chemistry,1
2,CORDIS,CORDIS_101040355,101040355,Probing (Orphan) Nuclear Receptors in Neurodeg...,"Neurodegenerative diseases such as Alzheimers,...",2022-05-01,2027-04-30,2022,1498813.0,1498813.0,...,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101040355,core_match,Other AI-enabled chemistry/materials,,0
3,CORDIS,CORDIS_101040729,101040729,FIrst NEar-TErm ApplicationS of QUAntum Devices,Quantum technologies have set remarkable miles...,2022-05-01,2027-04-30,2022,1485042.5,1485042.5,...,ERC STARTING GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101040729,core_match,AI-enabled chemistry,AI-enabled chemistry,1
4,CORDIS,CORDIS_101041177,101041177,Deciphering cellular and viral determinants of...,Herpes simplex virus 1 (HSV-1) is an important...,2022-05-01,2028-04-30,2022,1998008.75,1998008.75,...,ERC CONSOLIDATOR GRANTS,HORIZON-ERC,SIGNED,<NA>,<NA>,https://cordis.europa.eu/project/id/101041177,broad_match,Other AI-enabled chemistry/materials,,0


## Documenting the integrated data structure

A data dictionary is created so that every field in the final grant table has a clear and reproducible meaning.

For each column, the dictionary records:

- its table and column name;
- its final data type;
- its analytical definition;
- which sources provide the value;
- whether missing values are permitted;
- the transformation used to create it;
- important interpretation limitations.

The dictionary makes source-specific differences explicit. For example:

- `project_total_cost` is available only for CORDIS;
- `amount_obligated` is available only for NSF;
- `funding_mechanism` is available only for CORDIS;
- standardized activity classifications are available only for NSF;
- topic fields are derived for both sources using the shared rule-based taxonomy.

This documentation prevents source-specific fields from being interpreted as though they were directly comparable and provides a reference for later analysis, Tableau development, and Streamlit implementation.

In [26]:
# Create the data dictionary for the integrated grant table

grant_column_definitions = {
    "source": "Funding source that supplied the record.",
    "grant_key": "Globally unique source-prefixed grant identifier.",
    "source_id": "Original award or project identifier from the source.",
    "title": "Title of the funded award or project.",
    "abstract": "Main descriptive text for the funded activity.",
    "start_date": "Start date of the funded activity.",
    "end_date": "End date of the funded activity.",
    "award_year": "Calendar year in which the funded activity began.",
    "amount_native": "Primary funding amount in the source currency.",
    "project_total_cost": "Total project cost reported by CORDIS.",
    "amount_obligated": "Funding formally obligated to date by NSF.",
    "currency": "Native currency of the primary funding amount.",
    "organisation_name": "Lead, coordinating, or recipient organisation.",
    "organisation_city": "City of the lead or recipient organisation.",
    "organisation_country": "Country or country code reported by the source.",
    "programme_name": "Funding programme, call topic, or programme label.",
    "funding_mechanism": "Source-specific funding scheme or mechanism.",
    "status": "Source-reported or derived award status.",
    "activity_type": "Standardized activity category where available.",
    "source_activity_type": "Original source-specific activity classification.",
    "source_url": "Link to the original award or project record.",
    "relevance_tier": "Core or broad AI–chemistry/materials relevance tier.",
    "primary_topic": "Highest-priority taxonomy topic assigned to the grant.",
    "matched_topics": "All named taxonomy topics matched by the grant.",
    "topic_match_count": "Number of named taxonomy topics matched."
}


grant_source_availability = {
    "project_total_cost": "CORDIS only",
    "amount_obligated": "NSF only",
    "funding_mechanism": "CORDIS only",
    "activity_type": "NSF only",
    "source_activity_type": "NSF only",
    "primary_topic": "Derived for both sources",
    "matched_topics": "Derived for both sources",
    "topic_match_count": "Derived for both sources"
}


grant_transformations = {
    "grant_key": "CORDIS_ or NSF_ prefix added to the source identifier.",
    "abstract": "CORDIS objective and NSF abstract mapped to one field.",
    "start_date": "Parsed to datetime using source-specific ISO formats.",
    "end_date": "Parsed to datetime using source-specific ISO formats.",
    "award_year": "Derived from the standardized start date.",
    "amount_native": (
        "CORDIS uses ecMaxContribution; "
        "NSF uses estimatedTotalAmt."
    ),
    "currency": "Assigned as EUR for CORDIS and retained as USD for NSF.",
    "programme_name": (
        "CORDIS uses call_topic_title; "
        "NSF uses programme_name."
    ),
    "status": (
        "CORDIS retains source status; "
        "NSF Boolean status mapped to ACTIVE or INACTIVE."
    ),
    "primary_topic": (
        "Assigned using ordered rule-based keyword taxonomy."
    ),
    "matched_topics": (
        "Pipe-separated list of all rule-based topic matches."
    ),
    "topic_match_count": (
        "Count of named taxonomy topics matched by each grant."
    )
}


grant_limitations = {
    "amount_native": (
        "EUR and USD values must not be summed or directly compared "
        "without an exchange-rate method."
    ),
    "project_total_cost": (
        "CORDIS totalCost is incomplete or inconsistent for some projects "
        "and is not used as the primary award amount."
    ),
    "amount_obligated": (
        "Represents committed NSF funding, not necessarily the final "
        "estimated award value."
    ),
    "organisation_country": (
        "Country representation follows source reporting and may use "
        "country codes rather than full names."
    ),
    "funding_mechanism": (
        "No directly comparable NSF field was available."
    ),
    "activity_type": (
        "CORDIS nature was missing for all candidate projects."
    ),
    "source_activity_type": (
        "CORDIS nature was missing for all candidate projects."
    ),
    "primary_topic": (
        "Priority-based assignment; grants may legitimately match "
        "multiple topics."
    ),
    "matched_topics": (
        "Topic categories overlap, so counts across topics should not "
        "be added as unique-grant totals."
    ),
    "topic_match_count": (
        "A value of zero indicates the grant is retained in the "
        "fallback topic category."
    )
}


data_dictionary_rows = []

for column in grants_export_df.columns:

    data_dictionary_rows.append({
        "table_name": "grants_clean",
        "column_name": column,
        "data_type": str(
            grants_export_df[column].dtype
        ),
        "definition": grant_column_definitions[column],
        "source_availability": grant_source_availability.get(
            column,
            "CORDIS and NSF"
        ),
        "nullable": grants_export_df[column].isna().any(),
        "missing_count": int(
            grants_export_df[column].isna().sum()
        ),
        "transformation": grant_transformations.get(
            column,
            "Standardized from the corresponding source field."
        ),
        "limitations": grant_limitations.get(
            column,
            ""
        )
    })


data_dictionary_df = pd.DataFrame(
    data_dictionary_rows
)


print(
    "Data-dictionary rows:",
    len(data_dictionary_df)
)

print(
    "Matches final grant columns:",
    len(data_dictionary_df)
    == len(grants_export_df.columns)
)

print(
    "Undocumented definitions:",
    data_dictionary_df[
        "definition"
    ].isna().sum()
)

display(data_dictionary_df)

Data-dictionary rows: 25
Matches final grant columns: True
Undocumented definitions: 0


,table_name,column_name,data_type,definition,source_availability,nullable,missing_count,transformation,limitations
0,grants_clean,source,string,Funding source that supplied the record.,CORDIS and NSF,False,0,Standardized from the corresponding source field.,
1,grants_clean,grant_key,string,Globally unique source-prefixed grant identifier.,CORDIS and NSF,False,0,CORDIS_ or NSF_ prefix added to the source ide...,
2,grants_clean,source_id,string,Original award or project identifier from the ...,CORDIS and NSF,False,0,Standardized from the corresponding source field.,
3,grants_clean,title,string,Title of the funded award or project.,CORDIS and NSF,False,0,Standardized from the corresponding source field.,
4,grants_clean,abstract,string,Main descriptive text for the funded activity.,CORDIS and NSF,False,0,CORDIS objective and NSF abstract mapped to on...,
5,grants_clean,start_date,datetime64[ns],Start date of the funded activity.,CORDIS and NSF,False,0,Parsed to datetime using source-specific ISO f...,
6,grants_clean,end_date,datetime64[ns],End date of the funded activity.,CORDIS and NSF,False,0,Parsed to datetime using source-specific ISO f...,
7,grants_clean,award_year,Int64,Calendar year in which the funded activity began.,CORDIS and NSF,False,0,Derived from the standardized start date.,
8,grants_clean,amount_native,Float64,Primary funding amount in the source currency.,CORDIS and NSF,False,0,CORDIS uses ecMaxContribution; NSF uses estima...,EUR and USD values must not be summed or direc...
9,grants_clean,project_total_cost,Float64,Total project cost reported by CORDIS.,CORDIS only,True,2664,Standardized from the corresponding source field.,CORDIS totalCost is incomplete or inconsistent...


In [27]:
# Export the integrated funding and quality outputs

INTEGRATED_PROCESSED_DIR = (
    PROCESSED_DATA_DIR
    / "Integrated"
)

INTEGRATED_PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


integrated_output_paths = {
    "Integrated grant table": (
        INTEGRATED_PROCESSED_DIR
        / "grants_clean_2021_2025.csv"
    ),
    "Grant-topic bridge": (
        INTEGRATED_PROCESSED_DIR
        / "grant_topic_bridge_2021_2025.csv"
    ),
    "Complete funding topic-year summary": (
        INTEGRATED_PROCESSED_DIR
        / "grant_topic_year_summary_2021_2025.csv"
    ),
    "Three-source topic-year context": (
        INTEGRATED_PROCESSED_DIR
        / "topic_year_context_2021_2025.csv"
    ),
    "Data-quality report": (
        INTEGRATED_PROCESSED_DIR
        / "data_quality_report.csv"
    ),
    "Data dictionary": (
        INTEGRATED_PROCESSED_DIR
        / "data_dictionary.csv"
    ),
    "Schema crosswalk": (
        INTEGRATED_PROCESSED_DIR
        / "funding_schema_crosswalk.csv"
    ),
    "Topic taxonomy": (
        INTEGRATED_PROCESSED_DIR
        / "topic_mapping.csv"
    )
}


grants_export_df.to_csv(
    integrated_output_paths["Integrated grant table"],
    index=False,
    encoding="utf-8-sig"
)

grant_topic_bridge_df.to_csv(
    integrated_output_paths["Grant-topic bridge"],
    index=False,
    encoding="utf-8-sig"
)

grant_topic_year_complete_df.to_csv(
    integrated_output_paths[
        "Complete funding topic-year summary"
    ],
    index=False,
    encoding="utf-8-sig"
)

topic_year_context_df.to_csv(
    integrated_output_paths[
        "Three-source topic-year context"
    ],
    index=False,
    encoding="utf-8-sig"
)

data_quality_report_df.to_csv(
    integrated_output_paths["Data-quality report"],
    index=False,
    encoding="utf-8-sig"
)

data_dictionary_df.to_csv(
    integrated_output_paths["Data dictionary"],
    index=False,
    encoding="utf-8-sig"
)

schema_crosswalk_df.to_csv(
    integrated_output_paths["Schema crosswalk"],
    index=False,
    encoding="utf-8-sig"
)

topic_mapping_df.to_csv(
    integrated_output_paths["Topic taxonomy"],
    index=False,
    encoding="utf-8-sig"
)


for label, path in integrated_output_paths.items():

    print(f"\n{label}:")
    print(path)

    print(
        "File size:",
        f"{path.stat().st_size / 1_000_000:.2f} MB"
    )


Integrated grant table:
..\Data\Processed_Data\Integrated\grants_clean_2021_2025.csv
File size: 10.93 MB

Grant-topic bridge:
..\Data\Processed_Data\Integrated\grant_topic_bridge_2021_2025.csv
File size: 0.26 MB

Complete funding topic-year summary:
..\Data\Processed_Data\Integrated\grant_topic_year_summary_2021_2025.csv
File size: 0.01 MB

Three-source topic-year context:
..\Data\Processed_Data\Integrated\topic_year_context_2021_2025.csv
File size: 0.01 MB

Data-quality report:
..\Data\Processed_Data\Integrated\data_quality_report.csv
File size: 0.00 MB

Data dictionary:
..\Data\Processed_Data\Integrated\data_dictionary.csv
File size: 0.00 MB

Schema crosswalk:
..\Data\Processed_Data\Integrated\funding_schema_crosswalk.csv
File size: 0.00 MB

Topic taxonomy:
..\Data\Processed_Data\Integrated\topic_mapping.csv
File size: 0.00 MB


In [28]:
# Reload exported files and verify their saved structure

reloaded_outputs = {
    label: pd.read_csv(
        path,
        low_memory=False
    )
    for label, path in integrated_output_paths.items()
}

expected_output_shapes = {
    "Integrated grant table": (
        len(grants_export_df),
        len(grants_export_df.columns)
    ),
    "Grant-topic bridge": (
        len(grant_topic_bridge_df),
        len(grant_topic_bridge_df.columns)
    ),
    "Complete funding topic-year summary": (
        len(grant_topic_year_complete_df),
        len(grant_topic_year_complete_df.columns)
    ),
    "Three-source topic-year context": (
        len(topic_year_context_df),
        len(topic_year_context_df.columns)
    ),
    "Data-quality report": (
        len(data_quality_report_df),
        len(data_quality_report_df.columns)
    ),
    "Data dictionary": (
        len(data_dictionary_df),
        len(data_dictionary_df.columns)
    ),
    "Schema crosswalk": (
        len(schema_crosswalk_df),
        len(schema_crosswalk_df.columns)
    ),
    "Topic taxonomy": (
        len(topic_mapping_df),
        len(topic_mapping_df.columns)
    )
}


export_verification_rows = []

for label, reloaded_df in reloaded_outputs.items():

    expected_shape = expected_output_shapes[label]
    actual_shape = reloaded_df.shape

    export_verification_rows.append({
        "output": label,
        "expected_rows": expected_shape[0],
        "actual_rows": actual_shape[0],
        "expected_columns": expected_shape[1],
        "actual_columns": actual_shape[1],
        "shape_matches": actual_shape == expected_shape
    })


export_verification_df = pd.DataFrame(
    export_verification_rows
)

print(
    "Outputs successfully reloaded:",
    len(reloaded_outputs)
)

print(
    "Outputs with matching shapes:",
    export_verification_df[
        "shape_matches"
    ].sum(),
    "of",
    len(export_verification_df)
)

print(
    "Reloaded integrated grant duplicate keys:",
    reloaded_outputs[
        "Integrated grant table"
    ]["grant_key"].duplicated().sum()
)

print(
    "Reloaded topic-year duplicate rows:",
    reloaded_outputs[
        "Three-source topic-year context"
    ][
        ["topic", "year"]
    ].duplicated().sum()
)

display(export_verification_df)

Outputs successfully reloaded: 8
Outputs with matching shapes: 8 of 8
Reloaded integrated grant duplicate keys: 0
Reloaded topic-year duplicate rows: 0


,output,expected_rows,actual_rows,expected_columns,actual_columns,shape_matches
0,Integrated grant table,3339,3339,25,25,True
1,Grant-topic bridge,3245,3245,9,9,True
2,Complete funding topic-year summary,80,80,10,10,True
3,Three-source topic-year context,40,40,18,18,True
4,Data-quality report,20,20,10,10,True
5,Data dictionary,25,25,9,9,True
6,Schema crosswalk,20,20,4,4,True
7,Topic taxonomy,8,8,4,4,True


## Data integration and quality summary

The NSF, CORDIS, and OpenAlex datasets have been standardized and prepared for combined analysis in GrantScopeAI.

### Integrated funding table

NSF and CORDIS were transformed into a shared grant-level schema and appended into one dataset containing:

- 3,339 unique funding records;
- 675 CORDIS projects;
- 2,664 NSF awards;
- 25 final analytical columns;
- no duplicate global grant keys;
- no missing titles, abstracts, organisations, primary amounts, or source URLs;
- complete coverage within the 2021–2025 project period.

CORDIS funding is represented using maximum EU contribution in EUR. NSF funding is represented using estimated total award value in USD. Native currencies remain separate, and no unsupported cross-currency total is calculated.

### Shared topic taxonomy

Funding records were classified using the same eight research topics represented in OpenAlex:

- Autonomous laboratories;
- Reaction prediction;
- AI-enabled catalysis;
- Materials informatics;
- Cheminformatics;
- Molecular machine learning;
- AI-enabled materials;
- AI-enabled chemistry.

The rule-based taxonomy preserves all valid topic matches while assigning one priority-based primary topic.

Final topic coverage includes:

- 2,239 grants assigned to at least one named topic;
- 3,245 grant-topic relationships;
- 1,100 grants retained under `Other AI-enabled chemistry/materials`;
- no duplicate grant-topic relationships.

The fallback category is intentional and prevents weak or unsupported topic assignments.

### Topic-year analytical tables

A complete funding grid was created across:

- eight topics;
- five years from 2021 through 2025;
- two funding sources.

This produces 80 source-topic-year funding rows, including 19 explicit zero-grant combinations.

The funding summaries were then aligned with the 40-row OpenAlex publication-context table. The final three-source context table contains one row for each topic and year and presents:

- CORDIS grant activity and EUR funding;
- NSF grant activity and USD funding;
- OpenAlex publication counts and growth metrics.

OpenAlex remains separate from individual grants and is used only as topic-level research-momentum context.

### Data-quality results

The integrated quality report contains 20 validation rules.

- 19 rules returned `PASS`;
- one informational rule returned `REVIEW`.

The review result represents the 1,100 grants retained in the documented fallback topic category. It does not indicate failed data processing.

Validation confirmed:

- source row counts reconcile;
- global grant keys are unique;
- dates, amounts, currencies, relevance tiers, and URLs are valid;
- grant-topic relationships are unique;
- OpenAlex topic and year coverage is complete;
- the funding taxonomy aligns exactly with OpenAlex;
- the final topic-year context contains 40 unique records.

### Documentation and outputs

The notebook creates and exports:

- the integrated grant table;
- the grant-topic bridge;
- the complete funding topic-year summary;
- the three-source topic-year context table;
- the integrated data-quality report;
- the data dictionary;
- the NSF–CORDIS schema crosswalk;
- the topic taxonomy.

All eight exported files were successfully reloaded and matched their expected shapes.

The integrated datasets are now ready for exploratory analysis, Tableau development, similarity modelling, and Streamlit application development.